# HM-RAG + MAMMQA: runtime-optimized FULL pipeline

**Experiment:** `qa_full_runtime_optimized_v1_seed42`  
**Default:** `FAST_VALIDATION=True` runs exactly the first two frozen questions, then stops.  
**Scope:** engineering/runtime optimization only; no gold-answer tuning.

The scientific pipeline remains: question-only planner -> soft multimodal routing -> full-depth hybrid
retrieval -> graph expansion -> compact MAMMQA modality experts -> anchor-specific cross-modal synthesis
-> one verifier -> at most one targeted retrieval/reasoning round -> compact aggregator.

Runtime changes remove planner/Stage-I/Stage-II/verifier repair generations, retain only one optional
aggregator format repair, reduce reasoning context (not candidate depth), avoid repeated CUDA cache
clearing, compact verifier inputs and logs, add persistent-cache hooks, checkpoints, timings, ETA, and sharding.

In [1]:
# Cell 1 - environment diagnostics; allocator configured before importing torch
import importlib.util, os, platform, subprocess, sys
os.environ.setdefault("PYTORCH_ALLOC_CONF","expandable_segments:True")
print("Python:",sys.version)
try:
    import torch
    print("PyTorch:",torch.__version__,"CUDA:",torch.cuda.is_available(),"GPUs:",torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):print(i,torch.cuda.get_device_name(i))
except Exception as e:print("PyTorch diagnostic failed:",e)

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch: 2.10.0+cu128 CUDA: True GPUs: 2
0 Tesla T4
1 Tesla T4


In [2]:
# Cell 2 - install only missing, released packages
REQUIRED = {
    "transformers": "transformers>=4.49,<5", "accelerate": "accelerate>=1,<2",
    "bitsandbytes": "bitsandbytes>=0.45,<1", "qwen_vl_utils": "qwen-vl-utils>=0.0.8",
    "sentence_transformers": "sentence-transformers>=3,<6", "faiss": "faiss-cpu>=1.8,<2",
    "networkx": "networkx>=3,<4", "pandas": "pandas>=2,<3", "tabulate": "tabulate>=0.9,<1",
    "rank_bm25": "rank-bm25>=0.2.2,<1",
}
missing = [spec for module, spec in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else: print("All optional dependencies are already installed.")

Installing missing packages: ['bitsandbytes>=0.45,<1', 'qwen-vl-utils>=0.0.8', 'faiss-cpu>=1.8,<2', 'rank-bm25>=0.2.2,<1']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 54.2 MB/s eta 0:00:00


In [3]:
# Cell 3 - reproducible runtime configuration and isolated output/cache paths
import ast, copy, csv, gc, hashlib, json, math, os, pickle, random, re, string, time, zipfile
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Sequence, Tuple
import faiss, networkx as nx, numpy as np, pandas as pd, torch
from PIL import Image
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

NOTEBOOK_START_TIME=time.perf_counter()
SEED=42; RUN_MODE="smoke"; RUN_RETRIEVAL_ONLY=False; FAST_VALIDATION=False
NUM_SHARDS=1; SHARD_ID=0
EXPERIMENT_ID="qa_full_runtime_optimized_v3_seed42"
FAILED_EXPERIMENT_ID="qa_smoke16_full_pipeline_seed42"
MODEL_NAME="Qwen/Qwen2.5-VL-7B-Instruct"
EMBED_MODEL="BAAI/bge-small-en-v1.5"; CLIP_MODEL="openai/clip-vit-base-patch32"
RETRIEVAL_MODE="hybrid"; ROUTER_MODE="soft"
RETRIEVAL_CANDIDATE_K=20; MIN_METRIC_CANDIDATE_K=10; FALLBACK_RETRIEVAL_K=20
TEXT_CONTEXT_K=3; TABLE_CONTEXT_K=2; IMAGE_CONTEXT_K=2
FALLBACK_TEXT_K=1; FALLBACK_TABLE_K=1; FALLBACK_IMAGE_K=1
ROUTER_CONFIDENCE_THRESHOLD=0.65; PLANNER_MODALITY_THRESHOLD=0.55
RRF_K=60; GRAPH_SEED_K=10; GRAPH_MAX_EXPANDED_DOCS=50; GRAPH_MAX_DEGREE=50
TEXT_TOP_K=TEXT_CONTEXT_K; TABLE_TOP_K=TABLE_CONTEXT_K; IMAGE_TOP_K=IMAGE_CONTEXT_K; IMAGE_CANDIDATES=50
MAX_RETRIEVAL_ROUNDS=2; ENABLE_GRAPH=True; ENABLE_WEB=False
IMAGE_RETRIEVAL_MODE="hybrid"; AGGREGATOR_SEES_QUESTION=True
TABLE_CHUNK_ROWS=50; LARGE_TABLE_ROWS=120; IMAGE_REASONING_USES_TITLE=False

MAX_INPUT_TOKENS=4096
STAGE_MAX_NEW_TOKENS={"planner":64,"stage1_text":96,"stage1_table":96,"stage1_image":128,"stage2":128,"verifier":48,"aggregator":48}
FORMAT_REPAIR_NEW_TOKENS=48; REPAIR_STAGES={"aggregator"}
STAGE1_EVIDENCE_CHARS=1600; STAGE2_EVIDENCE_CHARS=1200; VERIFIER_EVIDENCE_CHARS=900
MAMMQA_TEMPERATURE=0.3; MAMMQA_TOP_P=0.7; MAMMQA_DO_SAMPLE=True
PLANNER_DO_SAMPLE=False; VERIFIER_DO_SAMPLE=False; IMAGE_PROCESSOR_USE_FAST=False
CLIP_GPU_MIN_FREE_GIB=6.0; BGE_GPU_MIN_FREE_GIB=3.0; CLIP_DEVICE="cpu"; BGE_DEVICE="cpu"

BASE_OUTPUT_ROOT=Path("/kaggle/working/hm_mammqa")
OUTPUT_ROOT=BASE_OUTPUT_ROOT/EXPERIMENT_ID
FAILED_OUTPUT_ROOT=BASE_OUTPUT_ROOT/FAILED_EXPERIMENT_ID
CACHE_EXPORT_ROOT=OUTPUT_ROOT/"persistent_cache_package"
DIRS={"cache":CACHE_EXPORT_ROOT/"cache","indexes":CACHE_EXPORT_ROOT/"indexes","images":BASE_OUTPUT_ROOT/"images",
      "predictions":OUTPUT_ROOT/"predictions","metrics":OUTPUT_ROOT/"metrics","logs":OUTPUT_ROOT/"logs",
      "diagnostics":OUTPUT_ROOT/"diagnostics","offload":OUTPUT_ROOT/"offload"}
DIRS["retrieval_logs"]=DIRS["diagnostics"]/"retrieval_logs"
for p in set(DIRS.values()):p.mkdir(parents=True,exist_ok=True)

input_root=Path("/kaggle/input")
# Set MAMMQA_PERSISTENT_CACHE_INPUT or edit this default to the mounted Kaggle cache dataset.
PERSISTENT_CACHE_INPUT=Path(os.environ.get("MAMMQA_PERSISTENT_CACHE_INPUT","/kaggle/input/hm-mammqa-persistent-cache"))
EXPECTED_CACHE_COUNTS={"texts":218285,"table_parents":10042,"images":57058}
CACHE_RESOURCE_STATUS={"image_manifest":"MISS","text":"MISS","table":"MISS","image_titles":"MISS","text_bm25":"MISS","table_bm25":"MISS","image_bm25":"MISS"}
STARTUP_TIMINGS={"image_manifest_seconds":0.0,"retrieval_cache_seconds":0.0,"qwen_load_seconds":0.0,"total_startup_seconds":0.0}
_CACHE_INVALID_REPORTED=set()

def cache_root_precheck(root):
    root=Path(root); metadata_path=root/"cache"/"cache_metadata.json"
    if not metadata_path.is_file():
        reason=f"{root}: missing cache/cache_metadata.json"
        if reason not in _CACHE_INVALID_REPORTED:print("CACHE INVALID:",reason); _CACHE_INVALID_REPORTED.add(reason)
        return False
    try:meta=json.loads(metadata_path.read_text(encoding="utf-8"))
    except Exception as exc:
        reason=f"{root}: unreadable metadata ({exc})"
        if reason not in _CACHE_INVALID_REPORTED:print("CACHE INVALID:",reason); _CACHE_INVALID_REPORTED.add(reason)
        return False
    dims=meta.get("index_dimensionality",{})
    checks=[
        (meta.get("embedding_model")==EMBED_MODEL,f"embedding model {meta.get('embedding_model')!r} != {EMBED_MODEL!r}"),
        (meta.get("document_counts")==EXPECTED_CACHE_COUNTS,f"document counts {meta.get('document_counts')!r} != {EXPECTED_CACHE_COUNTS!r}"),
        (all(isinstance(dims.get(k),int) and dims[k]>0 for k in ["text","table","image_titles"]),f"invalid index dimensions {dims!r}"),
        (bool(meta.get("dataset_fingerprint")),"missing dataset fingerprint"),
    ]
    failure=next((reason for ok,reason in checks if not ok),None)
    if failure:
        reason=f"{root}: {failure}"
        if reason not in _CACHE_INVALID_REPORTED:print("CACHE INVALID:",reason); _CACHE_INVALID_REPORTED.add(reason)
        return False
    return True

cache_roots=[]
def add_cache_candidate(candidate):
    candidate=Path(candidate)
    if candidate.exists() and ((candidate/"indexes").exists() or (candidate/"cache").exists()) and cache_root_precheck(candidate):cache_roots.append(candidate)

for candidate in [PERSISTENT_CACHE_INPUT,PERSISTENT_CACHE_INPUT/"persistent_cache_package"]:add_cache_candidate(candidate)
if input_root.exists():
    for mount in input_root.iterdir():
        for candidate in [mount,mount/"persistent_cache_package"]:add_cache_candidate(candidate)
        if mount.is_dir():
            for child in mount.iterdir():
                if child.is_dir():
                    add_cache_candidate(child); add_cache_candidate(child/"persistent_cache_package")
# Kaggle may mount datasets under /kaggle/input/datasets/<owner>/<dataset>.
# Bounded metadata globs find cache packages without recursively scanning the 57k image corpus.
if input_root.exists():
    for pattern in [
        "*/cache/cache_metadata.json","*/*/cache/cache_metadata.json","*/*/*/cache/cache_metadata.json",
        "*/persistent_cache_package/cache/cache_metadata.json","*/*/persistent_cache_package/cache/cache_metadata.json",
        "*/*/*/persistent_cache_package/cache/cache_metadata.json","*/*/*/*/persistent_cache_package/cache/cache_metadata.json",
    ]:
        for metadata_path in input_root.glob(pattern):
            root=metadata_path.parent.parent
            add_cache_candidate(root)
if BASE_OUTPUT_ROOT.exists():
    for experiment in BASE_OUTPUT_ROOT.iterdir():
        if experiment.is_dir():add_cache_candidate(experiment/"persistent_cache_package")
CACHE_READ_ROOTS=list(dict.fromkeys(p.resolve() for p in cache_roots))
print("PERSISTENT CACHE INPUT:",PERSISTENT_CACHE_INPUT)
print("VALID CACHE ROOTS:",[str(p) for p in CACHE_READ_ROOTS])

def artifact_group(specs):
    for root in CACHE_READ_ROOTS:
        paths={key:root/kind/name for key,kind,name in specs}
        if all(p.exists() for p in paths.values()):return paths,True,root
    paths={key:CACHE_EXPORT_ROOT/kind/name for key,kind,name in specs}
    for p in paths.values():p.parent.mkdir(parents=True,exist_ok=True)
    return paths,False,CACHE_EXPORT_ROOT

def artifact_output_group(specs):
    paths={key:CACHE_EXPORT_ROOT/kind/name for key,kind,name in specs}
    for p in paths.values():p.parent.mkdir(parents=True,exist_ok=True)
    return paths

GENERATION_DIAGNOSTICS_PATH=DIRS["logs"]/"generation_diagnostics.jsonl"
PLANNER_DIAGNOSTICS_PATH=DIRS["logs"]/"planner_diagnostics.jsonl"
TRUNCATION_DIAGNOSTICS_PATH=DIRS["logs"]/"truncation_diagnostics.jsonl"
TIMING_LOG_PATH=DIRS["logs"]/"timing.jsonl"
RUNTIME_CONFIG_PATH=DIRS["logs"]/"runtime_config.json"
OPTIMIZATION_SUMMARY_PATH=DIRS["logs"]/"runtime_optimization_summary.json"
INFORMATION_TRACE_PATH=DIRS["logs"]/"information_preservation_trace.jsonl"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
assert SEED==42 and RUN_MODE in {"smoke","full"} and RUN_RETRIEVAL_ONLY is False
assert RETRIEVAL_CANDIDATE_K>=MIN_METRIC_CANDIDATE_K>=10
assert MAX_RETRIEVAL_ROUNDS==2 and not ENABLE_WEB and IMAGE_REASONING_USES_TITLE is False
assert NUM_SHARDS>=1 and 0<=SHARD_ID<NUM_SHARDS and REPAIR_STAGES=={"aggregator"}

def gpu_memory(label):
    if not torch.cuda.is_available():print(label,"CPU only"); return []
    stats=[]; raw=[]
    for i in range(torch.cuda.device_count()):
        free,total=torch.cuda.mem_get_info(i); raw.append((free,total)); stats.append(f"GPU{i} free={free/2**30:.2f}/{total/2**30:.2f}GiB")
    print(label," | ".join(stats)); return raw
gpu_memory("startup:")

CACHE INVALID: /kaggle/working/hm_mammqa/qa_full_runtime_optimized_v3_seed42/persistent_cache_package: missing cache/cache_metadata.json
PERSISTENT CACHE INPUT: /kaggle/input/hm-mammqa-persistent-cache
VALID CACHE ROOTS: []
startup: GPU0 free=14.46/14.56GiB | GPU1 free=14.46/14.56GiB


[(15526068224, 15636037632), (15526068224, 15636037632)]

In [4]:
# Cell 4 - confirmed MMQA paths with unambiguous fallback
DATA_ROOT=Path("/kaggle/input/datasets/adibraian/multimodalqa-dataset")
NAMES={
 "train":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_train.jsonl",
 "dev":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_dev.jsonl",
 "test":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_test.jsonl",
 "texts":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_texts.jsonl",
 "tables":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_tables.jsonl",
 "images":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_images.jsonl"}
FOLDERS={k:f"MMQA_{k}.jsonl" for k in NAMES}
def resolve_dataset_file(key:str)->Path:
    primary=DATA_ROOT/FOLDERS[key]/NAMES[key]
    if primary.is_file(): return primary
    roots=[DATA_ROOT,Path("/kaggle/input")]
    matches=sorted({p.resolve() for root in roots if root.exists() for p in root.rglob(NAMES[key])})
    if len(matches)!=1: raise FileNotFoundError(f"{key}: expected one {NAMES[key]}, found {matches}")
    return matches[0]
PATHS={k:resolve_dataset_file(k) for k in NAMES}
for k,p in PATHS.items(): print(f"{k:7s} {p}")

train   /kaggle/input/datasets/adibraian/multimodalqa-dataset/MMQA_train.jsonl/multimodalqa_final_dataset_pipeline_camera_ready_MMQA_train.jsonl
dev     /kaggle/input/datasets/adibraian/multimodalqa-dataset/MMQA_dev.jsonl/multimodalqa_final_dataset_pipeline_camera_ready_MMQA_dev.jsonl
test    /kaggle/input/datasets/adibraian/multimodalqa-dataset/MMQA_test.jsonl/multimodalqa_final_dataset_pipeline_camera_ready_MMQA_test.jsonl
texts   /kaggle/input/datasets/adibraian/multimodalqa-dataset/MMQA_texts.jsonl/multimodalqa_final_dataset_pipeline_camera_ready_MMQA_texts.jsonl
tables  /kaggle/input/datasets/adibraian/multimodalqa-dataset/MMQA_tables.jsonl/multimodalqa_final_dataset_pipeline_camera_ready_MMQA_tables.jsonl
images  /kaggle/input/datasets/adibraian/multimodalqa-dataset/MMQA_images.jsonl/multimodalqa_final_dataset_pipeline_camera_ready_MMQA_images.jsonl


In [5]:
# Cell 5 - JSONL loading
def load_jsonl(path:Path)->List[dict]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for no,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except json.JSONDecodeError as e: raise ValueError(f"{path}:{no}: {e}") from e
    return out
dev_raw=load_jsonl(PATHS["dev"]); texts_raw=load_jsonl(PATHS["texts"])
tables_raw=load_jsonl(PATHS["tables"]); images_raw=load_jsonl(PATHS["images"])
print(len(dev_raw),len(texts_raw),len(tables_raw),len(images_raw))

2441 218285 10042 57058


In [6]:
# Cell 6 - schema inspection and hard inference/evaluation separation
for name,row in [("question",dev_raw[0]),("text",texts_raw[0]),("table",tables_raw[0]),("image",images_raw[0])]:
    print(name,{k:type(v).__name__ for k,v in row.items()})
@dataclass(frozen=True)
class QueryInput: qid:str; question:str
@dataclass(frozen=True)
class EvaluationOnly:
    gold_answers:List[str]; gold_support_ids:List[str]; gold_support_modalities:Dict[str,str]
    metadata_type:str="unknown"; gold_modalities:List[str]=field(default_factory=list)
    intermediate_answers:Any=None
FORBIDDEN={"answers","supporting_context","metadata","table_id","text_doc_ids","image_doc_ids",
           "modalities","type","intermediate_answers","gold_answers","gold_support_ids"}
def assert_safe_query(q:QueryInput):
    assert isinstance(q,QueryInput) and set(asdict(q))=={"qid","question"}
    assert not (set(asdict(q)) & FORBIDDEN)
def query_only(row:dict)->QueryInput:
    q=QueryInput(str(row.get("qid",row.get("id",""))),str(row["question"]))
    assert_safe_query(q); return q
def assert_inference_boundary(fn_name:str,q:QueryInput):
    assert_safe_query(q)
    assert set(asdict(q))=={"qid","question"},f"{fn_name} received a non-query field"
print("LEAKAGE CHECK PASSED")

question {'qid': 'str', 'question': 'str', 'answers': 'list', 'metadata': 'dict', 'supporting_context': 'list'}
text {'title': 'str', 'url': 'str', 'id': 'str', 'text': 'str'}
table {'title': 'str', 'url': 'str', 'id': 'str', 'table': 'dict'}
image {'title': 'str', 'url': 'str', 'id': 'str', 'path': 'str'}
LEAKAGE CHECK PASSED


In [7]:
# Cell 7 - cached official-image manifest, ZIP fallback and pixel validation
def safe_extract(zp:Path,dst:Path):
    with zipfile.ZipFile(zp) as z:
        root=dst.resolve()
        for member in z.infolist():
            target=(dst/member.filename).resolve()
            if root not in target.parents and target!=root:raise ValueError(f"Unsafe ZIP member: {member.filename}")
        z.extractall(dst)

def load_image_manifest(records):
    paths,found,root=artifact_group([("manifest","cache","image_manifest.json")])
    manifest_path=paths["manifest"]
    if found:
        try:
            payload=json.loads(manifest_path.read_text(encoding="utf-8")); mapping={}
            for image_id,rel in payload.get("paths",{}).items():
                p=input_root/rel
                if p.is_file():mapping[str(image_id)]=p
            if payload.get("count")==57058 and len(payload.get("paths",{}))==57058 and len(mapping)==len(records)==57058:
                CACHE_RESOURCE_STATUS["image_manifest"]="HIT"; print("CACHE HIT: image_manifest",manifest_path); return mapping
            print("CACHE INVALID:",root,"image manifest count/path mismatch")
        except Exception as e:print("CACHE INVALID:",root,"image manifest unreadable:",e)
    return None

def discover_image_paths(records):
    cached=load_image_manifest(records)
    if cached is not None:return cached
    expected={Path(str(r.get("path",""))).name for r in records if r.get("path")}
    mounts=input_root; extract=DIRS["images"]/"official"
    named=[] if not mounts.exists() else [p for p in mounts.rglob("final_dataset_images") if p.is_dir()]
    zips=[] if not mounts.exists() else [p for p in mounts.rglob("*.zip") if re.search(r"(mmqa|multimodal).*image|image.*(mmqa|multimodal)",p.name,re.I)]
    valid_zips=[]
    for z in zips:
        try:
            with zipfile.ZipFile(z) as zh:
                if any(Path(n).name in expected for n in zh.namelist()):valid_zips.append(z)
        except zipfile.BadZipFile:pass
    if not named and valid_zips:
        if len(valid_zips)>1:raise RuntimeError(f"Ambiguous official image ZIPs: {valid_zips}")
        marker=extract/".complete"
        if not marker.exists():extract.mkdir(parents=True,exist_ok=True); safe_extract(valid_zips[0],extract); marker.touch()
        named=[extract]
    roots=named or ([mounts] if mounts.exists() else []); by_name={}
    for root in roots:
        for p in root.rglob("*"):
            if p.is_file() and p.name in expected:
                if p.name in by_name and by_name[p.name]!=p:raise RuntimeError(f"Ambiguous image filename {p.name}")
                by_name[p.name]=p
    mapping={str(r["id"]):by_name[Path(str(r.get("path",""))).name] for r in records if Path(str(r.get("path",""))).name in by_name}
    if len(mapping)==len(records) and all(input_root==p or input_root in p.parents for p in mapping.values()):
        out=CACHE_EXPORT_ROOT/"cache"/"image_manifest.json"; out.parent.mkdir(parents=True,exist_ok=True)
        out.write_text(json.dumps({"count":len(mapping),"paths":{k:str(v.relative_to(input_root)) for k,v in mapping.items()}},indent=2),encoding="utf-8")
        print("Created publishable image manifest:",out)
    CACHE_RESOURCE_STATUS["image_manifest"]="BUILT"
    return mapping

_image_manifest_started=time.perf_counter()
IMAGE_PATHS=discover_image_paths(images_raw)
STARTUP_TIMINGS["image_manifest_seconds"]=time.perf_counter()-_image_manifest_started
assert len(images_raw)==57058,f"Expected 57,058 official MMQA images, found metadata for {len(images_raw)}"
assert len(IMAGE_PATHS)==57058,f"Expected 57,058 resolved official images, found {len(IMAGE_PATHS)}"
for p in list(IMAGE_PATHS.values())[:5]:
    with Image.open(p) as im:im.verify()
print(f"official image manifest verified: {len(IMAGE_PATHS):,}/57,058; sample PIL checks=5")
FULL_IMAGE_BENCHMARK=True

Created publishable image manifest: /kaggle/working/hm_mammqa/qa_full_runtime_optimized_v3_seed42/persistent_cache_package/cache/image_manifest.json
official image manifest verified: 57,058/57,058; sample PIL checks=5


In [8]:
# Cell 8 - corpus normalization and table representations
def cell_text(c:Any)->str:
    if isinstance(c,dict): return str(c.get("text",c.get("value","")))
    return str(c)
def table_parts(row:dict)->Tuple[List[str],List[List[str]]]:
    t=row.get("table",row); header=t.get("header",[]); rows=t.get("table_rows",t.get("rows",[]))
    heads=[str(x.get("column_name",x.get("text",""))) if isinstance(x,dict) else str(x) for x in header]
    return heads,[[cell_text(c) for c in rr] for rr in rows]
def markdown_table(heads,rows)->str:
    width=max([len(heads),*(len(r) for r in rows)] or [0]); heads=(heads+[f"col_{i}" for i in range(len(heads),width)])[:width]
    clean=lambda s:str(s).replace("|","\\|").replace("\n"," ")
    return "\n".join(["| "+" | ".join(map(clean,heads))+" |","| "+" | ".join(["---"]*width)+" |",*["| "+" | ".join(map(clean,r+[""]*(width-len(r))))+" |" for r in rows]])
def chunk_table_rows(headers:List[str],rows:List[List[str]],max_rows=TABLE_CHUNK_ROWS,max_chars=6000):
    chunks=[]; start=0
    while start<len(rows):
        end=start; size=len(" | ".join(headers))
        while end<len(rows) and end-start<max_rows:
            row_size=sum(len(str(v)) for v in rows[end])+len(headers)*4
            if end>start and size+row_size>max_chars: break
            size+=row_size; end+=1
        chunks.append((start,max(start+1,end)))
        start=max(start+1,end)
    return chunks or [(0,0)]
def normalize_corpus():
    docs=[]
    for r in texts_raw:
        did=str(r["id"]); title=str(r.get("title","")); body=str(r.get("text",r.get("paragraph","")))
        docs.append({"doc_id":did,"chunk_id":did,"parent_doc_id":did,"parent_table_id":None,"modality":"text","title":title,"retrieval_text":f"{title}\n{body}","reasoning_content":body,"raw_content":body,"image_path":None,"row_start":None,"row_end":None,"headers":[]})
    for r in tables_raw:
        did=str(r["id"]); title=str(r.get("title",r.get("name",""))); h,rows=table_parts(r)
        spans=chunk_table_rows(h,rows)
        for start,end in spans:
            ch=rows[start:end]; cid=did if len(spans)==1 else f"{did}::rows_{start}-{max(start,end-1)}"
            linear=" ; ".join([f"{k}: {v}" for rr in ch for k,v in zip(h,rr)])
            neighbor_start=max(0,start-1); neighbor_end=min(len(rows),end+1)
            reasoning=markdown_table(h,rows[neighbor_start:neighbor_end])
            docs.append({"doc_id":cid,"chunk_id":cid,"parent_doc_id":did,"parent_table_id":did,"modality":"table","title":title,"retrieval_text":f"{title}\ncolumns: {' | '.join(h)}\n{linear}","reasoning_content":reasoning,"raw_content":reasoning,"image_path":None,"source_record":r,"row_start":start,"row_end":max(start,end-1),"headers":h})
    for r in images_raw:
        did=str(r["id"]); title=str(r.get("title",""))
        docs.append({"doc_id":did,"chunk_id":did,"parent_doc_id":did,"parent_table_id":None,"modality":"image","title":title,"retrieval_text":title,"reasoning_content":{"path":r.get("path")},"raw_content":{"path":r.get("path"),"url":r.get("url")},"image_path":str(IMAGE_PATHS[did]) if did in IMAGE_PATHS else None,"row_start":None,"row_end":None,"headers":[]})
    return docs
CORPUS=normalize_corpus(); BY_MODALITY={m:[d for d in CORPUS if d["modality"]==m] for m in ["text","table","image"]}
DOC_LOOKUP={(d["modality"],d["doc_id"]):d for d in CORPUS}
PARENT_LOOKUP=defaultdict(list)
for d in CORPUS:PARENT_LOOKUP[(d["modality"],d["parent_doc_id"])].append(d)
TABLE_PARENT_TO_CHUNKS={pid:chunks for (mod,pid),chunks in PARENT_LOOKUP.items() if mod=="table"}
DATASET_FINGERPRINT=hashlib.sha256(json.dumps({
    "counts":{"texts":len(texts_raw),"table_parents":len(tables_raw),"images":len(images_raw)},
    "boundary_ids":{"text":[str(texts_raw[0]["id"]),str(texts_raw[-1]["id"])],
                    "table":[str(tables_raw[0]["id"]),str(tables_raw[-1]["id"])],
                    "image":[str(images_raw[0]["id"]),str(images_raw[-1]["id"])]}
},sort_keys=True).encode()).hexdigest()
CACHE_CREATION_VERSION="hm_mammqa_cache_v2"
def cache_metadata_compatible(root):
    if not cache_root_precheck(root):return False
    path=Path(root)/"cache"/"cache_metadata.json"; meta=json.loads(path.read_text(encoding="utf-8"))
    if meta.get("dataset_fingerprint")!=DATASET_FINGERPRINT:
        print("CACHE INVALID:",root,"dataset fingerprint mismatch"); return False
    if meta.get("creation_version") not in {"hm_mammqa_cache_v1",CACHE_CREATION_VERSION}:
        print("CACHE INVALID:",root,"unsupported creation version",meta.get("creation_version")); return False
    return True

print({m:len(v) for m,v in BY_MODALITY.items()},"table parents",len(TABLE_PARENT_TO_CHUNKS))
assert sum(len(v) for v in TABLE_PARENT_TO_CHUNKS.values())==len(BY_MODALITY["table"])
if not RUN_RETRIEVAL_ONLY:
    assert len(texts_raw)>200000 and len(tables_raw)>10000 and len(images_raw)>50000,"The full MMQA corpus was not loaded"
    assert len(BY_MODALITY["text"])==len(texts_raw) and len(TABLE_PARENT_TO_CHUNKS)==len(tables_raw) and len(BY_MODALITY["image"])==len(images_raw)
    print(f"FULL CORPUS CHECK PASSED: {len(texts_raw):,} text documents, {len(tables_raw):,} original tables, {len(images_raw):,} official images")

{'text': 218285, 'table': 10082, 'image': 57058} table parents 10042
FULL CORPUS CHECK PASSED: 218,285 text documents, 10,042 original tables, 57,058 official images


In [9]:
# Cells 9-10 - persistent normalized FAISS/embedding/BM25 indexes
_retrieval_cache_started=time.perf_counter()
def fingerprint(records):
    h=hashlib.sha256(); h.update(EMBED_MODEL.encode())
    for d in records:h.update((d["doc_id"]+"\0"+d["retrieval_text"]).encode("utf-8"))
    return h.hexdigest()

class DenseIndex:
    def __init__(self,modality,records,encoder):self.modality,self.records,self.encoder=modality,records,encoder; self.index=None
    def build_or_load(self):
        specs=[("index","indexes",f"{self.modality}_faiss.index"),("mapping","indexes",f"{self.modality}_doc_mapping.json"),("embeddings","cache",f"{self.modality}_embeddings.npy")]
        paths,found,root=artifact_group(specs); fp=fingerprint(self.records)
        mapping=json.loads(paths["mapping"].read_text()) if found else {}
        if found and cache_metadata_compatible(root) and mapping.get("fingerprint")==fp:
            candidate=faiss.read_index(str(paths["index"])); metadata=json.loads((Path(root)/"cache"/"cache_metadata.json").read_text(encoding="utf-8"))
            expected_dim=metadata.get("index_dimensionality",{}).get(self.modality)
            if candidate.ntotal==len(self.records) and candidate.d==expected_dim and len(mapping.get("ids",[]))==len(self.records):
                self.index=candidate; CACHE_RESOURCE_STATUS[self.modality]="HIT"; print(f"CACHE HIT: {self.modality}",root); return
            print(f"CACHE INVALID: {root}: {self.modality} index/mapping count or dimension mismatch")
        if found:paths=artifact_output_group(specs)
        emb=self.encoder.encode([d["retrieval_text"] for d in self.records],batch_size=128,normalize_embeddings=True,show_progress_bar=True).astype("float32")
        np.save(paths["embeddings"],emb); self.index=faiss.IndexFlatIP(emb.shape[1]); self.index.add(emb); faiss.write_index(self.index,str(paths["index"]))
        paths["mapping"].write_text(json.dumps({"fingerprint":fp,"ids":[d["doc_id"] for d in self.records]}))
        CACHE_RESOURCE_STATUS[self.modality]="BUILT"; print(f"Built {self.modality} cache package under",CACHE_EXPORT_ROOT)
    def search(self,query,k,subquery="",query_source="original_question"):
        q=self.encoder.encode([query],normalize_embeddings=True).astype("float32"); scores,idx=self.index.search(q,min(k,len(self.records)))
        return [{**self.records[i],"dense_rank":rank,"dense_score":float(s),"bm25_rank":None,"bm25_score":None,"hybrid_rrf_score":0.0,"graph_rank":None,"graph_score":0.0,"final_score":float(s),"retrieved_by_subquery":[subquery or query],"query_provenance":[{"query_text":subquery or query,"query_source":query_source,"rank":rank,"score":float(s),"method":"dense"}]} for rank,(s,i) in enumerate(zip(scores[0],idx[0]),1) if i>=0]

def bm25_tokens(text):return re.findall(r"[a-z0-9]+",str(text).lower())
class BM25Index:
    def __init__(self,modality,records):self.modality,self.records=modality,records; self.index=None
    def build_or_load(self):
        specs=[("bm25","indexes",f"{self.modality}_bm25.pkl")]; paths,found,root=artifact_group(specs); fp=fingerprint(self.records)
        if found:
            if cache_metadata_compatible(root):
                payload=pickle.loads(paths["bm25"].read_bytes())
                if payload.get("fingerprint")==fp and len(getattr(payload.get("index"),"doc_len",[]))==len(self.records):
                    self.index=payload["index"]; CACHE_RESOURCE_STATUS[f"{self.modality}_bm25"]="HIT"; print(f"CACHE HIT: {self.modality}_bm25",root); return
                print(f"CACHE INVALID: {root}: {self.modality} BM25 fingerprint/count mismatch")
            paths=artifact_output_group(specs)
        self.index=BM25Okapi([bm25_tokens(d["retrieval_text"]) for d in self.records]); paths["bm25"].write_bytes(pickle.dumps({"fingerprint":fp,"index":self.index},pickle.HIGHEST_PROTOCOL)); CACHE_RESOURCE_STATUS[f"{self.modality}_bm25"]="BUILT"
    def search(self,query,k,subquery="",query_source="original_question"):
        scores=np.asarray(self.index.get_scores(bm25_tokens(query))); k=min(k,len(scores)); idx=np.argpartition(scores,-k)[-k:] if 0<k<len(scores) else np.arange(len(scores)); idx=idx[np.argsort(scores[idx])[::-1]][:k]
        return [{**self.records[i],"dense_rank":None,"dense_score":None,"bm25_rank":rank,"bm25_score":float(scores[i]),"hybrid_rrf_score":0.0,"graph_rank":None,"graph_score":0.0,"final_score":float(scores[i]),"retrieved_by_subquery":[subquery or query],"query_provenance":[{"query_text":subquery or query,"query_source":query_source,"rank":rank,"score":float(scores[i]),"method":"bm25"}]} for rank,i in enumerate(idx,1)]

gpu_memory("before BGE:")
QUERY_ENCODER=SentenceTransformer(EMBED_MODEL,device="cuda:0" if torch.cuda.is_available() else "cpu")
TEXT_INDEX=DenseIndex("text",BY_MODALITY["text"],QUERY_ENCODER); TEXT_INDEX.build_or_load()
TABLE_INDEX=DenseIndex("table",BY_MODALITY["table"],QUERY_ENCODER); TABLE_INDEX.build_or_load()
IMAGE_TITLE_INDEX=DenseIndex("image_titles",BY_MODALITY["image"],QUERY_ENCODER); IMAGE_TITLE_INDEX.build_or_load()
QUERY_ENCODER.to("cpu"); BGE_DEVICE="cpu"
for index in [TEXT_INDEX,TABLE_INDEX,IMAGE_TITLE_INDEX]:index.encoder=QUERY_ENCODER
BM25_INDEXES={m:BM25Index(m,BY_MODALITY[m]) for m in ["text","table","image"]}
for index in BM25_INDEXES.values():index.build_or_load()
CACHE_METADATA={"embedding_model":EMBED_MODEL,"document_counts":{"texts":len(texts_raw),"table_parents":len(tables_raw),"images":len(images_raw)},
 "dataset_fingerprint":DATASET_FINGERPRINT,"dataset_paths":{k:str(v) for k,v in PATHS.items()},
 "index_dimensionality":{x.modality:int(x.index.d) for x in [TEXT_INDEX,TABLE_INDEX,IMAGE_TITLE_INDEX]},"creation_version":CACHE_CREATION_VERSION}
(CACHE_EXPORT_ROOT/"cache"/"cache_metadata.json").write_text(json.dumps(CACHE_METADATA,indent=2),encoding="utf-8")
STARTUP_TIMINGS["retrieval_cache_seconds"]=time.perf_counter()-_retrieval_cache_started
print("RETRIEVAL CACHE STATUS:",{k:CACHE_RESOURCE_STATUS[k] for k in ["text","table","image_titles","text_bm25","table_bm25","image_bm25"]})

def rrf_fuse(dense,lexical,k):
    merged={}
    for source,items in [("dense",dense),("bm25",lexical)]:
        for rank,d in enumerate(items,1):
            key=d["doc_id"]; x=merged.setdefault(key,{**d,"dense_rank":None,"dense_score":None,"bm25_rank":None,"bm25_score":None,"retrieved_by_subquery":[],"query_provenance":[]})
            x[f"{source}_rank"]=rank; x[f"{source}_score"]=d.get(f"{source}_score")
            x["retrieved_by_subquery"]=list(dict.fromkeys(x["retrieved_by_subquery"]+d.get("retrieved_by_subquery",[])))
            x["query_provenance"] += [p for p in d.get("query_provenance",[]) if p not in x["query_provenance"]]
    for x in merged.values():
        x["hybrid_rrf_score"]=(1/(RRF_K+x["dense_rank"]) if x["dense_rank"] else 0)+(1/(RRF_K+x["bm25_rank"]) if x["bm25_rank"] else 0); x["final_score"]=x["hybrid_rrf_score"]
    return sorted(merged.values(),key=lambda x:x["hybrid_rrf_score"],reverse=True)[:k]

def search_modality(modality,query,k,mode,subquery,query_source="original_question"):
    dense_index={"text":TEXT_INDEX,"table":TABLE_INDEX,"image":IMAGE_TITLE_INDEX}[modality]; internal=max(k*3,50)
    if mode=="dense":return dense_index.search(query,k,subquery,query_source)
    if mode=="bm25":return BM25_INDEXES[modality].search(query,k,subquery,query_source)
    return rrf_fuse(dense_index.search(query,internal,subquery,query_source),BM25_INDEXES[modality].search(query,internal,subquery,query_source),k)

before BGE: GPU0 free=14.46/14.56GiB | GPU1 free=14.46/14.56GiB


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (10 outputs left out)

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Built table cache package under /kaggle/working/hm_mammqa/qa_full_runtime_optimized_v3_seed42/persistent_cache_package


Batches:   0%|          | 0/446 [00:00<?, ?it/s]

Built image_titles cache package under /kaggle/working/hm_mammqa/qa_full_runtime_optimized_v3_seed42/persistent_cache_package
RETRIEVAL CACHE STATUS: {'text': 'BUILT', 'table': 'BUILT', 'image_titles': 'BUILT', 'text_bm25': 'BUILT', 'table_bm25': 'BUILT', 'image_bm25': 'BUILT'}


In [10]:
# Cell 11 - official-image CLIP reranking with persistent cache and safe GPU fallback
class ClipReranker:
    def __init__(self):self.model=None; self.processor=None; self.cache={}; self.actual_device="cpu"
    @staticmethod
    def unwrap_features(output):
        if isinstance(output,torch.Tensor):return output
        pooled=getattr(output,"pooler_output",None)
        if isinstance(pooled,torch.Tensor):return pooled
        if isinstance(output,(tuple,list)):
            tensors=[x for x in output if isinstance(x,torch.Tensor) and x.ndim==2]
            if tensors:return tensors[-1]
        raise TypeError(f"Unsupported CLIP feature output: {type(output).__name__}")
    def load(self):
        from transformers import CLIPModel,CLIPProcessor
        self.processor=CLIPProcessor.from_pretrained(CLIP_MODEL,use_fast=False)
        target=CLIP_DEVICE
        try:self.model=CLIPModel.from_pretrained(CLIP_MODEL).eval().to(target); self.actual_device=target
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache(); self.model=CLIPModel.from_pretrained(CLIP_MODEL).eval().to("cpu"); self.actual_device="cpu"
        specs=[("clip","cache","clip_embeddings.pkl")]; paths,found,root=artifact_group(specs)
        if found and cache_metadata_compatible(root):
            payload=pickle.loads(paths["clip"].read_bytes())
            if isinstance(payload,dict) and payload.get("clip_model")==CLIP_MODEL and payload.get("dataset_fingerprint")==DATASET_FINGERPRINT:
                self.cache=payload.get("embeddings",{}); print("CACHE HIT: clip_embeddings",root)
            else:print("CACHE REJECTED: clip_embeddings metadata mismatch",root)
        self.cache_path=artifact_output_group(specs)["clip"]
        print("CLIP actual device =",self.actual_device)
    def encode_missing(self,docs):
        if self.model is None:self.load()
        missing=[d for d in docs if d.get("image_path") and d["doc_id"] not in self.cache]
        for start in range(0,len(missing),32):
            batch=missing[start:start+32]; ims=[]
            for d in batch:
                with Image.open(d["image_path"]) as im:ims.append(im.convert("RGB"))
            x=self.processor(images=ims,return_tensors="pt").to(self.model.device)
            with torch.inference_mode():z=self.unwrap_features(self.model.get_image_features(**x)); z=z/z.norm(dim=-1,keepdim=True)
            self.cache.update({d["doc_id"]:e.cpu().numpy().astype("float32") for d,e in zip(batch,z)}); del x,z
        if missing:self.cache_path.write_bytes(pickle.dumps({"clip_model":CLIP_MODEL,"dataset_fingerprint":DATASET_FINGERPRINT,"embeddings":self.cache},pickle.HIGHEST_PROTOCOL))
    def rerank(self,query,docs,k):
        docs=[d for d in docs if d.get("image_path")]; self.encode_missing(docs)
        if not docs:return []
        text_limit=int(getattr(self.model.config.text_config,"max_position_embeddings",77))
        x=self.processor(text=[query],return_tensors="pt",padding=True,truncation=True,max_length=text_limit).to(self.model.device)
        with torch.inference_mode():q=self.unwrap_features(self.model.get_text_features(**x)); q=q/q.norm(dim=-1,keepdim=True)
        q=q[0].cpu().numpy(); del x; out=[]
        for d in docs:out.append({**d,"clip_score":float(np.dot(q,self.cache[d["doc_id"]])),"final_score":float(np.dot(q,self.cache[d["doc_id"]]))})
        return sorted(out,key=lambda x:x["final_score"],reverse=True)[:k]
CLIP=ClipReranker()

In [11]:
# Cell 12 - original-table graph and bounded cross-modal expansion
def norm_entity(x): return re.sub(r"\s+"," ",str(x).strip().lower())
def cell_entities(cell:Any)->List[str]:
    values=[cell_text(cell)]
    if isinstance(cell,dict):
        for key in ["links","entities","wiki_links"]:
            for link in cell.get(key,[]) or []:
                if isinstance(link,dict):values.extend([link.get("title",""),link.get("text","")])
                else:values.append(link)
    return list(dict.fromkeys("e::"+norm_entity(v) for v in values if 1<len(norm_entity(v))<=100 and not norm_entity(v).replace(".","").isdigit()))
def build_graph():
    paths,found,root=artifact_group([("graph","cache","mmqa_graph_v2.pkl")]); path=paths["graph"]; fp="v2-"+fingerprint(CORPUS)
    if found:
        payload=pickle.loads(path.read_bytes())
        if payload["fingerprint"]==fp:return payload["graph"]
        path=artifact_output_group([("graph","cache","mmqa_graph_v2.pkl")])["graph"]
    g=nx.Graph()
    for modality,raws in [("text",texts_raw),("image",images_raw)]:
        for raw in raws:
            did=str(raw["id"]); dn=f"d::{modality}::{did}"; title=norm_entity(raw.get("title",""))
            g.add_node(dn,kind="document",doc_id=did,modality=modality)
            if title:g.add_edge(dn,"e::"+title,kind="title")
    for raw in tables_raw:
        did=str(raw["id"]); dn=f"d::table::{did}"; title=norm_entity(raw.get("title",raw.get("name","")))
        g.add_node(dn,kind="document",doc_id=did,modality="table")
        if title:g.add_edge(dn,"e::"+title,kind="title")
        table=raw.get("table",raw); rows=table.get("table_rows",table.get("rows",[]))
        for row in rows:
            entities=list(dict.fromkeys(e for cell in row for e in cell_entities(cell)))[:20]
            for entity in entities:g.add_edge(dn,entity,kind="cell")
            for i,a in enumerate(entities):
                for b in entities[i+1:i+4]:g.add_edge(a,b,kind="row")
    path.write_bytes(pickle.dumps({"fingerprint":fp,"graph":g},pickle.HIGHEST_PROTOCOL)); return g
GRAPH=build_graph() if ENABLE_GRAPH else None
class WebRetriever:
    """Extension point only; web evidence is forbidden in official MMQA experiments."""
    def search(self,query:str,k:int=5):
        if not ENABLE_WEB: raise RuntimeError("Web retrieval is disabled for reproducibility and leakage control")
        raise NotImplementedError("Provide a versioned, auditable backend before enabling non-benchmark experiments")
def representative_record(modality:str,parent_id:str)->Optional[dict]:
    records=PARENT_LOOKUP.get((modality,parent_id),[])
    return records[0] if records else None
def cross_modal_graph_expand(seed_candidates:Dict[str,List[dict]])->Tuple[List[dict],List[dict]]:
    if not GRAPH:return [],[]
    seeds=sorted([d for xs in seed_candidates.values() for d in xs],key=lambda x:x.get("final_score",0),reverse=True)[:GRAPH_SEED_K]
    found={}; provenance=[]; seed_ids={(d["modality"],d["parent_doc_id"]) for d in seeds}
    for seed_rank,seed in enumerate(seeds,1):
        start=f'd::{seed["modality"]}::{seed["parent_doc_id"]}'
        if start not in GRAPH:continue
        queue=[(start,0,[start])]; visited={start}
        while queue:
            node,hop,path=queue.pop(0)
            if hop>=2:continue
            neighbors=sorted(GRAPH.neighbors(node),key=str)[:GRAPH_MAX_DEGREE]
            for nxt in neighbors:
                if nxt in visited:continue
                visited.add(nxt); new_path=path+[nxt]; new_hop=hop+1
                attrs=GRAPH.nodes[nxt]
                if attrs.get("kind")=="document":
                    key=(attrs["modality"],attrs["doc_id"])
                    if key not in seed_ids:
                        score=1/((RRF_K+seed_rank)*(new_hop+1)); old=found.get(key)
                        rec=representative_record(*key)
                        if rec and (old is None or score>old["graph_score"]):
                            found[key]={**rec,"dense_rank":None,"dense_score":None,"bm25_rank":None,"bm25_score":None,"hybrid_rrf_score":0.0,"graph_score":score,"final_score":score,"retrieved_by_subquery":seed.get("retrieved_by_subquery",[])}
                        if len(provenance)<GRAPH_MAX_EXPANDED_DOCS*4:
                            provenance.append({"seed_doc":seed["parent_doc_id"],"seed_modality":seed["modality"],"entity_path":new_path[1:-1],"expanded_doc":attrs["doc_id"],"expanded_modality":attrs["modality"],"hop_count":new_hop,"graph_score":score})
                queue.append((nxt,new_hop,new_path))
    ranked=sorted(found.values(),key=lambda x:x["graph_score"],reverse=True)[:GRAPH_MAX_EXPANDED_DOCS]
    for rank,d in enumerate(ranked,1):d["graph_rank"]=rank
    return ranked,provenance

In [12]:
# Cell 13 - deterministic question-only planner for retrieval-only experiments
def question_only_plan(question:str)->dict:
    q=question.lower(); confidence={"text":0.50,"table":0.45,"image":0.20}
    if re.search(r"\b(color|colour|logo|pictured|shown|image|photo|looks?|wearing|flag|shape)\b",q):confidence["image"]=0.88
    if re.search(r"\b(how many|number|year|when|date|rank|score|record|role|list|first|last|most|least)\b",q):confidence["table"]=0.78
    if re.search(r"\b(according|described|stated|why|who|biography|history|where)\b",q):confidence["text"]=0.72
    relational=bool(re.search(r"\b(where|which|whose|that|who)\b.*\b(where|which|what|when|color|colour|year)\b",q))
    if relational:
        confidence["table"]=max(confidence["table"],0.70); confidence["text"]=max(confidence["text"],0.60)
    available=["text","table"]+(["image"] if IMAGE_PATHS else [])
    selected=[m for m in available if confidence[m]>=PLANNER_MODALITY_THRESHOLD]
    if not selected:selected=[max(available,key=lambda m:confidence[m])]
    routing_confidence=max(confidence[m] for m in selected)
    return {"needs_decomposition":False,"subqueries":[question],"needed_modalities":selected,"modality_confidence":confidence,"routing_confidence":routing_confidence,"needs_relational_search":relational,"planner_backend":"question_only_rules"}
def fallback_plan(question):return question_only_plan(question)

In [13]:
# Cell 14 - retrieval memory checkpoint
if CLIP.model is not None: CLIP.model.to("cpu")
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
gpu_memory("retrieval stack ready:")

retrieval stack ready: GPU0 free=14.41/14.56GiB | GPU1 free=14.46/14.56GiB


[(15475736576, 15636037632), (15526068224, 15636037632)]

In [14]:
# Cell 15 - memory-bounded Qwen load, then safe small-encoder placement
_qwen_load_started=time.perf_counter()
QWEN_CALL_COUNTER=defaultdict(int); FORMAT_REPAIR_CALL_COUNTER=defaultdict(int); model=None; processor=None
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
gpu_memory("before Qwen:")
processor=AutoProcessor.from_pretrained(MODEL_NAME,min_pixels=256*28*28,max_pixels=384*28*28,use_fast=IMAGE_PROCESSOR_USE_FAST)
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
max_memory={i:"10GiB" for i in range(torch.cuda.device_count())}; max_memory["cpu"]="24GiB"
model=Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_NAME,quantization_config=bnb,device_map="balanced",max_memory=max_memory,
    offload_folder=str(DIRS["offload"]),offload_state_dict=True,low_cpu_mem_usage=True,attn_implementation="sdpa").eval()
assert model is not None and processor is not None
STARTUP_TIMINGS["qwen_load_seconds"]=time.perf_counter()-_qwen_load_started

if torch.cuda.is_available():
    free0,_=torch.cuda.mem_get_info(0)
    if free0/2**30>=BGE_GPU_MIN_FREE_GIB:
        QUERY_ENCODER.to("cuda:0"); BGE_DEVICE="cuda:0"
        for index in [TEXT_INDEX,TABLE_INDEX,IMAGE_TITLE_INDEX]:index.encoder=QUERY_ENCODER
    free0,_=torch.cuda.mem_get_info(0)
    if free0/2**30>=CLIP_GPU_MIN_FREE_GIB:CLIP_DEVICE="cuda:0"
print("QWEN LOADED; BGE device =",BGE_DEVICE,"; CLIP requested device =",CLIP_DEVICE)
print("MAMMQA: do_sample =",MAMMQA_DO_SAMPLE,"temperature =",MAMMQA_TEMPERATURE,"top_p =",MAMMQA_TOP_P)
print("stage_max_new_tokens =",STAGE_MAX_NEW_TOKENS,"repair_stages =",REPAIR_STAGES)
gpu_memory("post-placement:")

before Qwen: GPU0 free=14.41/14.56GiB | GPU1 free=14.46/14.56GiB


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

... (4 outputs left out)

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

QWEN LOADED; BGE device = cuda:0 ; CLIP requested device = cuda:0
MAMMQA: do_sample = True temperature = 0.3 top_p = 0.7
stage_max_new_tokens = {'planner': 64, 'stage1_text': 96, 'stage1_table': 96, 'stage1_image': 128, 'stage2': 128, 'verifier': 48, 'aggregator': 48} repair_stages = {'aggregator'}
post-placement: GPU0 free=12.78/14.56GiB | GPU1 free=10.20/14.56GiB


[(13726711808, 15636037632), (10956374016, 15636037632)]

In [15]:
# Cell 16 - compact parsers, protected evidence budgeting, aggregator-only repair and OOM recovery
def append_jsonl_log(path,row):
    with Path(path).open("a",encoding="utf-8") as f:
        f.write(json.dumps(row,ensure_ascii=False,default=str)+"\n"); f.flush(); os.fsync(f.fileno())

def clip01(x,default=0.0):
    try:return max(0.0,min(1.0,float(x)))
    except:return default

def stable_call_seed(qid,stage,retry_count):return (SEED+int(hashlib.sha256(f"{qid}:{stage}:{retry_count}".encode()).hexdigest()[:8],16))%(2**31-1)
def evidence_id(item):return str(item.get("evidence_id",item.get("doc_id","unknown")))
def public_evidence(item):return {k:item.get(k) for k in ["evidence_id","modality","content"] if item.get(k) is not None}

def build_messages(system,protected,evidence):
    content=[]
    for item in evidence:
        if item.get("modality")=="image" and item.get("image_path") and Path(item["image_path"]).is_file():content.append({"type":"image","image":f'file://{item["image_path"]}'})
    content.append({"type":"text","text":json.dumps({"protected":protected,"evidence":[public_evidence(x) for x in evidence]},ensure_ascii=False)})
    return [{"role":"system","content":system},{"role":"user","content":content}]

def processor_inputs(messages):
    prompt=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True); images,videos=process_vision_info(messages)
    kwargs={"text":[prompt],"return_tensors":"pt","padding":True,"truncation":False}
    if images:kwargs.update(images=images,videos=videos)
    inputs=processor(**kwargs); return inputs,int(inputs["input_ids"].shape[-1]),len(images or [])

def shorten_lowest_ranked_text_evidence(evidence):
    for index in sorted(range(len(evidence)),key=lambda i:(evidence[i].get("rank",0),i),reverse=True):
        content=evidence[index].get("content")
        if isinstance(content,str) and len(content)>500:
            new=copy.deepcopy(evidence); new[index]["content"]=content[:max(500,len(content)//2)]+"\n[shortened]"; return new,True
    return evidence,False

def prepare_bounded_inputs(system,protected,evidence,max_budget=MAX_INPUT_TOKENS):
    work=copy.deepcopy(evidence); removed=[]; trimmed=False
    while True:
        messages=build_messages(system,protected,work); inputs,count,image_count=processor_inputs(messages)
        if count<=max_budget:return inputs,count,image_count,removed,trimmed
        del inputs
        shortened,did=shorten_lowest_ranked_text_evidence(work)
        if did:work=shortened; trimmed=True; continue
        if work:
            index=max(range(len(work)),key=lambda i:(work[i].get("rank",0),i)); removed.append(evidence_id(work[index])); work=work[:index]+work[index+1:]; trimmed=True; continue
        raise RuntimeError(f"PROTECTED_PROMPT_TOO_LARGE: {count}>{max_budget}")

def tag_value(text,name):
    closed=re.search(fr"<{name}>\s*(.*?)\s*</{name}>",str(text),re.I|re.S)
    if closed:return closed.group(1).strip()
    partial=re.search(fr"<{name}>\s*(.*?)(?=<[a-z_]+>|$)",str(text),re.I|re.S)
    return partial.group(1).strip() if partial else ""

def labelled_value(text,name):
    match=re.search(fr"^\s*{name}\s*[:=]\s*(.*?)\s*$",str(text),re.I|re.M)
    return match.group(1).strip() if match else ""

def parse_tagged_agent(raw,stage):
    # Stage-I uses compact lines; legacy XML remains accepted for the already-healthy Stage-II contract.
    answer=labelled_value(raw,"ANSWER") or tag_value(raw,"answer")
    insights=labelled_value(raw,"INSIGHTS?") or tag_value(raw,"insights")
    subanswer=labelled_value(raw,"SUBANSWER") or tag_value(raw,"subanswer") or "NONE"
    ids=labelled_value(raw,"EVIDENCE_IDS?") or tag_value(raw,"evidence_ids")
    if not answer or not insights:return None
    parsed={"insights":insights,"candidate_answer":answer,"subanswer":subanswer,
            "supporting_doc_ids":[x.strip() for x in re.split(r"[,;]",ids) if x.strip() and x.strip().upper()!="NONE"]}
    if stage.startswith("stage1_"):
        expected=stage.split("_",1)[1]
        modality=(labelled_value(raw,"MODALITY") or tag_value(raw,"modality") or expected).lower()
        if modality!=expected:return None
        parsed["modality"]=modality
    return parsed

def parse_stage1_output(raw,modality):
    text=str(raw or "").strip(); strict=parse_tagged_agent(text,f"stage1_{modality}") if text else None
    answer=labelled_value(text,"ANSWER") or tag_value(text,"answer") or None
    insight=labelled_value(text,"INSIGHTS?") or tag_value(text,"insights") or text
    subanswer=labelled_value(text,"SUBANSWER") or tag_value(text,"subanswer") or None
    supplied=labelled_value(text,"MODALITY") or tag_value(text,"modality") or modality
    ids=labelled_value(text,"EVIDENCE_IDS?") or tag_value(text,"evidence_ids")
    return {"usable":bool(text),"schema_parse_success":strict is not None,"modality":supplied.lower() if supplied else modality,
            "candidate_answer":answer,"insights":insight,"subanswer":subanswer,
            "supporting_doc_ids":[x.strip() for x in re.split(r"[,;]",ids) if x.strip() and x.strip().upper()!="NONE"],
            "raw_output":raw}

def parse_compact_planner(raw):
    modalities_match=re.search(r"^\s*MODALITIES\s*:\s*([^\n]+)",str(raw),re.I|re.M)
    if not modalities_match:return None
    modalities=[m.lower() for m in re.findall(r"text|table|image",modalities_match.group(1),re.I)]
    if not modalities:return None
    queries=[]
    for value in re.findall(r"^\s*QUERY(?:[123])?\s*:\s*([^\n]+)",str(raw),re.I|re.M):
        value=value.strip()
        if value and value.lower() not in {"none","n/a"}:queries.append(value)
    weights={m:1/3 for m in ["text","table","image"]}; wm=re.search(r"^\s*WEIGHTS\s*:\s*([^\n]+)",str(raw),re.I|re.M)
    if wm:
        found={m.lower():float(v) for m,v in re.findall(r"(text|table|image)\s*=\s*(0(?:\.\d+)?|1(?:\.0+)?)",wm.group(1),re.I)}
        if found:
            total=sum(found.values()) or 1.0; weights={m:found.get(m,0.0)/total for m in ["text","table","image"]}
    return {"needed_modalities":list(dict.fromkeys(modalities)),"subqueries":queries[:3],"modality_confidence":weights}

def parse_compact_verifier(raw):
    lines=[line.strip() for line in str(raw).splitlines() if line.strip()]
    if not lines:return None
    labelled=re.fullmatch(r"STATUS\s*[:=]\s*(supported|insufficient|conflict)",lines[0],re.I)
    bare=re.fullmatch(r"(supported|insufficient|conflict)",lines[0],re.I)
    match=labelled or bare
    if not match:return None
    status=match.group(1).lower()
    if status=="supported":
        if len(lines)!=1:return None
        return {"status":status,"best_answer":"","retry_modality":"none","retry_query":"","feedback_possible":False}
    if len(lines)>3:return None
    fields={}
    for line in lines[1:]:
        field=re.fullmatch(r"(RETRY_MODALITY|RETRY_QUERY)\s*[:=]\s*(.*?)",line,re.I)
        if not field:return None
        key=field.group(1).upper()
        if key in fields:return None
        fields[key]=field.group(2).strip()
    mod=fields.get("RETRY_MODALITY","none").lower(); query=fields.get("RETRY_QUERY","")
    if mod not in {"text","table","image","none"}:return None
    if query.upper() in {"NONE","N/A"}:query=""
    possible=mod in {"text","table","image"} and bool(query)
    return {"status":status,"best_answer":"","retry_modality":mod,"retry_query":query,"feedback_possible":possible}

def parse_rejection_reason(raw,stage):
    if stage.startswith("stage1_"):
        expected=stage.split("_",1)[1]
        if not (labelled_value(raw,"ANSWER") or tag_value(raw,"answer")):return "missing ANSWER"
        if not (labelled_value(raw,"INSIGHTS?") or tag_value(raw,"insights")):return "missing INSIGHT"
        supplied=labelled_value(raw,"MODALITY") or tag_value(raw,"modality")
        if supplied and supplied.lower()!=expected:return f"wrong MODALITY: expected {expected}, got {supplied}"
        return "unrecognized Stage-I structure"
    if stage=="verifier":
        lines=[line.strip() for line in str(raw).splitlines() if line.strip()]
        if not lines:return "empty verifier output"
        if not re.fullmatch(r"(?:STATUS\s*[:=]\s*)?(?:supported|insufficient|conflict)",lines[0],re.I):return "missing or invalid STATUS"
        return "unexpected, duplicate, or invalid verifier field"
    return "parser contract mismatch"

def parse_compact_aggregator(raw):
    answer=tag_value(raw,"answer")
    if not answer:
        match=re.search(r"^\s*ANSWER\s*:\s*([^\n]+)",str(raw),re.I|re.M); answer=match.group(1).strip() if match else ""
    if not answer:return None
    return {"answer":answer,"status":"abstain" if answer.strip().upper()=="ABSTAIN" else "answered"}

def stage_budget(stage):
    base=stage.replace("_format_repair","")
    key=base if base in STAGE_MAX_NEW_TOKENS else ("stage2" if base.startswith("stage2_") else base)
    return STAGE_MAX_NEW_TOKENS[key]

def generate_once(system,protected,evidence,qid,stage,do_sample,retry_count=0):
    max_new_tokens=stage_budget(stage); inputs,input_count,image_count,removed,trimmed=prepare_bounded_inputs(system,protected,evidence)
    seed=stable_call_seed(qid,stage,retry_count); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    inputs=inputs.to(model.device); kwargs={"max_new_tokens":max_new_tokens,"do_sample":do_sample}
    if do_sample:kwargs.update(temperature=MAMMQA_TEMPERATURE,top_p=MAMMQA_TOP_P)
    else:kwargs.update(temperature=None,top_p=None)
    started=time.perf_counter(); oom_retried=False
    print(f"[QWEN START] qid={qid} stage={stage} input={input_count} images={image_count} max_new={max_new_tokens}",flush=True)
    try:
        QWEN_CALL_COUNTER[qid]+=1
        with torch.inference_mode():ids=model.generate(**inputs,**kwargs)
    except torch.cuda.OutOfMemoryError:
        oom_retried=True; gc.collect(); torch.cuda.empty_cache(); print(f"[OOM RETRY] qid={qid} stage={stage}",flush=True)
        try:
            QWEN_CALL_COUNTER[qid]+=1
            with torch.inference_mode():ids=model.generate(**inputs,**kwargs)
        except torch.cuda.OutOfMemoryError:
            append_jsonl_log(GENERATION_DIAGNOSTICS_PATH,{"qid":qid,"stage":stage,"oom":True,"oom_retry_failed":True,"time":time.time()}); del inputs; raise
    output_ids=ids[:,inputs.input_ids.shape[1]:]; raw=processor.batch_decode(output_ids,skip_special_tokens=True)[0].strip()
    meta={"qid":qid,"stage":stage,"input_token_count":input_count,"max_input_budget":MAX_INPUT_TOKENS,"image_count":image_count,
          "output_token_count":int(output_ids.shape[-1]),"max_new_tokens":max_new_tokens,"retry_count":retry_count,
          "generation_time":time.perf_counter()-started,"context_trimmed":trimmed,"removed_evidence_ids":removed,
          "protected_prompt_truncated":False,"seed":seed,"oom_retried":oom_retried}
    print(f"[QWEN DONE] qid={qid} stage={stage} output={meta['output_token_count']} seconds={meta['generation_time']:.1f}",flush=True)
    del inputs,ids,output_ids
    return raw,meta

REPAIR_PROMPT="""Reformat the previous output only. Do not add new evidence or facts. Return exactly <answer>short final answer OR ABSTAIN</answer>."""

def stage_call(system,protected,evidence,qid,stage,do_sample,parser,allow_repair=False):
    assert allow_repair==(stage in REPAIR_STAGES)
    raw,meta=generate_once(system,protected,evidence,qid,stage,do_sample,0); parsed=parser(raw); ok=parsed is not None
    meta["parse_success"]=ok; meta["raw_output"]=raw; meta["parse_rejection_reason"]=None if ok else parse_rejection_reason(raw,stage); append_jsonl_log(GENERATION_DIAGNOSTICS_PATH,meta)
    if meta["context_trimmed"]:append_jsonl_log(TRUNCATION_DIAGNOSTICS_PATH,meta)
    repair_raw=None; diagnostics=[meta]
    if not ok and allow_repair:
        FORMAT_REPAIR_CALL_COUNTER[qid]+=1
        repair_raw,rmeta=generate_once(REPAIR_PROMPT,{"previous_output":raw[:2500]},[],qid,stage+"_format_repair",False,1)
        parsed=parser(repair_raw); ok=parsed is not None; rmeta["parse_success"]=ok; append_jsonl_log(GENERATION_DIAGNOSTICS_PATH,rmeta); diagnostics.append(rmeta)
    print(f"[PARSE] qid={qid} stage={stage} success={ok} repair={repair_raw is not None}",flush=True)
    return {"raw_output":raw,"repair_raw_output":repair_raw,"parsed_output":parsed if ok else None,"parse_success":ok,
            "retry_count":int(repair_raw is not None),"generation_schema_failure":not ok,"generation_diagnostics":diagnostics}

def compact_or_raw(envelope,max_chars=1600):
    if envelope and envelope.get("parse_success"):return envelope["parsed_output"]
    if envelope and envelope.get("usable_output"):
        usable=dict(envelope["usable_parsed_output"]); usable["raw_output"]=str(usable.get("raw_output",""))[:max_chars]; return usable
    return {"schema_failure":True,"usable":False,"raw_output":str((envelope or {}).get("raw_output",""))[:max_chars]}

def canonical_answer(value):
    answer=str(value or "").strip().strip('"\''); low=answer.lower().strip(" .")
    if low in {"yes","yeah","true"}:return "yes"
    if low in {"no","false"}:return "no"
    if re.fullmatch(r"[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?",answer):return answer.replace(",","")
    return answer

def parse_final_answer(envelope):
    parsed=(envelope or {}).get("parsed_output") if (envelope or {}).get("parse_success") else None
    if parsed and str(parsed.get("answer","")).strip():return canonical_answer(parsed["answer"]),True
    for text in [(envelope or {}).get("raw_output",""),(envelope or {}).get("repair_raw_output","")]:
        answer=tag_value(text,"answer")
        if answer:return canonical_answer(answer),True
    return "",False

In [16]:
# Cell 17 - compact one-call planner with deterministic safe fallback
PLANNER_PROMPT="""Plan retrieval from the question only. Return at most these lines, with no explanation:
MODALITIES: comma-separated text,table,image
WEIGHTS: text=0.xx,table=0.xx,image=0.xx
QUERY1: short retrieval query
QUERY2: optional short retrieval query
QUERY3: optional short retrieval query"""

def normalize_subquery(value):
    if isinstance(value,str):return value.strip()
    if isinstance(value,dict):
        for key in ["query","subquery","question","text"]:
            if isinstance(value.get(key),str) and value[key].strip():return value[key].strip()
    return ""
def assert_clean_retrieval_query(query):
    stripped=str(query).lstrip(); assert not stripped.startswith(("{'query':",'{"query":')); assert not re.search(r"['\"]modality_confidence['\"]\s*:",str(query))
def build_query_records(question,subqueries):
    records=[]; seen=set(); original=str(question).strip()
    for value,source in [(original,"original_question")]+[(x,"decomposed_subquery") for x in subqueries]:
        text=original if source=="original_question" else normalize_subquery(value); key=" ".join(text.split()).casefold()
        if text and key not in seen:assert_clean_retrieval_query(text); records.append({"text":text,"source":source}); seen.add(key)
    assert records[0]=={"text":original,"source":"original_question"}; return records

def safe_planner_fallback(question,reason):
    mods=["text","table"]+(["image"] if IMAGE_PATHS else []); weights={m:1/len(mods) for m in mods}
    return {"needed_modalities":mods,"modality_confidence":{m:weights.get(m,0.0) for m in ["text","table","image"]},
            "routing_confidence":max(weights.values()),"normalized_subqueries":[],"retrieval_queries":build_query_records(question,[]),
            "planner_parse_failure":True,"fallback_reason":reason}

def fallback_plan(question,reason="planner_disabled"):
    plan=safe_planner_fallback(question,reason); plan.update(raw_planner_output=None,parsed_planner_output=None,planner_parse_success=True,planner_schema_failure=False,planner_parse_failure=False,planner_generation=None)
    return plan

def plan_query(q):
    assert_safe_query(q); started=time.perf_counter()
    envelope=stage_call(PLANNER_PROMPT,{"qid":q.qid,"question":q.question},[],q.qid,"planner",PLANNER_DO_SAMPLE,parse_compact_planner,False)
    parsed=envelope.get("parsed_output")
    if parsed:
        normalized=[normalize_subquery(x) for x in parsed.get("subqueries",[])]; normalized=[x for x in normalized if x][:3]
        mods=[m for m in parsed["needed_modalities"] if m in {"text","table","image"} and (m!="image" or IMAGE_PATHS)]
        if not mods:mods=["text","table"]+(["image"] if IMAGE_PATHS else [])
        conf={m:clip01(parsed.get("modality_confidence",{}).get(m),0.0) for m in ["text","table","image"]}
        plan={"needed_modalities":mods,"modality_confidence":conf,"routing_confidence":max(conf.values()),"normalized_subqueries":normalized,
              "retrieval_queries":build_query_records(q.question,normalized),"planner_parse_failure":False}
    else:plan=safe_planner_fallback(q.question,"planner_parse_failure")
    plan.update(raw_planner_output=envelope["raw_output"],parsed_planner_output=parsed,planner_parse_success=envelope["parse_success"],
                planner_schema_failure=envelope["generation_schema_failure"],planner_generation=envelope,planner_backend="qwen_compact",
                planner_latency_seconds=time.perf_counter()-started)
    return plan

In [17]:
# Cell 18 - soft-priority routing, full candidate retrieval and separate context selection
def ensure_encoder():
    assert QUERY_ENCODER is not None,"Persistent query encoder was not initialized"; return QUERY_ENCODER

def route_budgets(plan,router_mode):
    assert router_mode in {"hard","soft","all"}
    available=["text","table"]+(["image"] if IMAGE_PATHS else [])
    normal={"text":TEXT_CONTEXT_K,"table":TABLE_CONTEXT_K,"image":IMAGE_CONTEXT_K}
    fallback={"text":FALLBACK_TEXT_K,"table":FALLBACK_TABLE_K,"image":FALLBACK_IMAGE_K}
    selected=set(plan.get("needed_modalities",[]))&set(available)
    if not selected:selected={max(available,key=lambda m:plan.get("modality_confidence",{}).get(m,0))}
    low_conf=float(plan.get("routing_confidence",0))<ROUTER_CONFIDENCE_THRESHOLD
    # Soft routing is prioritization only: every modality is searched at the full candidate depth.
    if router_mode in {"soft","all"}:
        candidates={m:max(RETRIEVAL_CANDIDATE_K,MIN_METRIC_CANDIDATE_K) for m in available}
        if router_mode=="all" or low_conf:contexts={m:normal[m] for m in available}
        else:contexts={m:(normal[m] if m in selected else fallback[m]) for m in available}
        return candidates,contexts
    return ({m:max(RETRIEVAL_CANDIDATE_K,MIN_METRIC_CANDIDATE_K) for m in selected},{m:normal[m] for m in selected})

def merge_subquery_rankings(groups,k):
    merged={}
    for group in groups:
        for rank,d in enumerate(group,1):
            key=d["doc_id"]; x=merged.setdefault(key,{**d,"subquery_rrf_score":0.0,"retrieved_by_subquery":[],"query_provenance":[]})
            x["subquery_rrf_score"]+=1/(RRF_K+rank)
            x["retrieved_by_subquery"]=list(dict.fromkeys(x["retrieved_by_subquery"]+d.get("retrieved_by_subquery",[])))
            x["query_provenance"]+= [p for p in d.get("query_provenance",[]) if p not in x["query_provenance"]]
            for field in ["dense_score","bm25_score","hybrid_rrf_score"]:
                values=[v for v in [x.get(field),d.get(field)] if v is not None]; x[field]=max(values) if values else None
    for x in merged.values():x["final_score"]=x["subquery_rrf_score"]
    return sorted(merged.values(),key=lambda x:x["final_score"],reverse=True)[:k]

def aggregate_parent_candidates(items,k):
    groups=defaultdict(list)
    for d in items:groups[d["parent_doc_id"]].append(d)
    parents=[]
    for parent_id,chunks in groups.items():
        chunks=sorted(chunks,key=lambda x:x.get("final_score",0),reverse=True); best=dict(chunks[0])
        best["matched_chunks"]=[{"chunk_id":x["doc_id"],"row_start":x.get("row_start"),"row_end":x.get("row_end"),"score":x.get("final_score",0)} for x in chunks[:5]]
        best["retrieved_by_subquery"]=list(dict.fromkeys(q for x in chunks for q in x.get("retrieved_by_subquery",[])))
        best["query_provenance"]=[p for x in chunks for p in x.get("query_provenance",[])]
        parents.append(best)
    return sorted(parents,key=lambda x:x.get("final_score",0),reverse=True)[:k]

def fuse_graph(base,expanded,k_by_mod):
    out={m:list(base.get(m,[])) for m in ["text","table","image"]}; graph_by_mod=defaultdict(list)
    for d in expanded:graph_by_mod[d["modality"]].append(d)
    for mod in out:
        base_rank={d["parent_doc_id"]:i+1 for i,d in enumerate(out[mod])}; graph_rank={d["parent_doc_id"]:i+1 for i,d in enumerate(graph_by_mod[mod])}
        records={d["parent_doc_id"]:dict(d) for d in out[mod]}
        for d in graph_by_mod[mod]:records.setdefault(d["parent_doc_id"],dict(d))
        fused=[]
        for did,d in records.items():
            d["graph_rank"]=graph_rank.get(did); d["graph_score"]=next((x.get("graph_score",0) for x in graph_by_mod[mod] if x["parent_doc_id"]==did),0)
            d["final_score"]=(1/(RRF_K+base_rank[did]) if did in base_rank else 0)+(1/(RRF_K+graph_rank[did]) if did in graph_rank else 0)
            if d["graph_rank"]:d.setdefault("query_provenance",[]).append({"query_text":"GRAPH_EXPANSION","query_source":"graph_expansion","rank":d["graph_rank"],"score":d["graph_score"],"method":"graph"})
            fused.append(d)
        out[mod]=sorted(fused,key=lambda x:x["final_score"],reverse=True)[:k_by_mod.get(mod,RETRIEVAL_CANDIDATE_K)]
    return out

def select_context(question,candidates,context_budgets):
    context={m:[] for m in ["text","table","image"]}
    for mod,k in context_budgets.items():
        pool=candidates.get(mod,[])
        context[mod]=CLIP.rerank(question,[d for d in pool if d.get("image_path")],k) if mod=="image" and pool else pool[:k]
    return context

def retrieve_evidence(q,plan,retrieval_mode=RETRIEVAL_MODE,router_mode=ROUTER_MODE,use_graph=True,fixed_all=False):
    assert_safe_query(q); ensure_encoder(); started=time.perf_counter()
    if fixed_all:router_mode="all"
    candidate_budgets,context_budgets=route_budgets(plan,router_mode)
    query_records=plan.get("retrieval_queries") or build_query_records(q.question,plan.get("normalized_subqueries",[]))
    assert query_records[0]=={"text":q.question,"source":"original_question"}
    candidates={m:[] for m in ["text","table","image"]}
    for mod,k in candidate_budgets.items():
        groups=[search_modality(mod,qr["text"],max(k,RETRIEVAL_CANDIDATE_K),retrieval_mode,qr["text"],qr["source"]) for qr in query_records]
        chunks=merge_subquery_rankings(groups,max(k*3,RETRIEVAL_CANDIDATE_K)); candidates[mod]=aggregate_parent_candidates(chunks,k)
        assert len(candidates[mod])>=min(MIN_METRIC_CANDIDATE_K,len(BY_MODALITY[mod]))
    graph_provenance=[]
    if use_graph:
        expanded,graph_provenance=cross_modal_graph_expand(candidates)
        limits={m:max(candidate_budgets.get(m,0),MIN_METRIC_CANDIDATE_K) for m in candidates}
        candidates=fuse_graph(candidates,expanded,limits)
    context=select_context(q.question,candidates,context_budgets)
    priorities=sorted(({"modality":m,"confidence":float(plan.get("modality_confidence",{}).get(m,0)),"candidate_budget":candidate_budgets[m],"context_budget":context_budgets.get(m,0)} for m in candidate_budgets),key=lambda x:(-x["confidence"],x["modality"]))
    diagnostic={"qid":q.qid,"question":q.question,"raw_planner_output":plan.get("raw_planner_output"),
                "parsed_planner_output":plan.get("parsed_planner_output"),"normalized_subqueries":plan.get("normalized_subqueries",[]),
                "retrieval_queries":query_records,"modality_confidence":plan.get("modality_confidence",{}),
                "modalities_searched":list(candidate_budgets),"candidate_counts":{m:len(xs) for m,xs in candidates.items()},
                "context_counts":{m:len(xs) for m,xs in context.items()}}
    append_jsonl_log(PLANNER_DIAGNOSTICS_PATH,diagnostic)
    return {"retrieval_candidates":candidates,"context_evidence":context,"graph_provenance":graph_provenance,
            "candidate_ids":{m:[d["parent_doc_id"] for d in xs] for m,xs in candidates.items()},
            "context_ids":{m:[d["parent_doc_id"] for d in xs] for m,xs in context.items()},
            "routing":{"mode":router_mode,"retrieval_mode":retrieval_mode,"candidate_budgets":candidate_budgets,"context_budgets":context_budgets,"searched_modalities":list(candidate_budgets),"soft_router_priorities":priorities},
            "retrieval_latency_seconds":time.perf_counter()-started}

In [18]:
# Cell 19 - compact Stage-I modality experts; no repair generation
EXPERT_PROMPT="""You are the modality-specific MAMMQA expert. Use only the supplied modality evidence.
Return exactly four compact lines, putting the answer first:
MODALITY: text | table | image
ANSWER: short candidate answer OR INSUFFICIENT
INSIGHT: maximum 2 concise sentences
SUBANSWER: optional short answer or NONE
No XML, confidence, evidence IDs, long explanation, or outside knowledge."""

def clean_evidence(items,modality,content_chars):
    clean=[]
    for rank,d in enumerate(items,1):
        body=d.get("reasoning_content",d.get("raw_content",""))
        if modality=="image":body="Official MultiModalQA image pixels are attached."
        elif not isinstance(body,str):body=json.dumps(body,ensure_ascii=False,default=str)
        clean.append({"evidence_id":str(d["parent_doc_id"]),"modality":modality,"content":body[:content_chars],
                      "image_path":d.get("image_path") if modality=="image" else None,"rank":rank})
    return clean

def run_modality_experts(q,evidence):
    assert_inference_boundary("run_modality_experts",q); out={}
    for mod in ["text","table","image"]:
        items=[d for d in evidence.get(mod,[]) if mod!="image" or d.get("image_path")]
        if not items:continue
        ev=clean_evidence(items,mod,STAGE1_EVIDENCE_CHARS); allowed={x["evidence_id"] for x in ev}
        envelope=stage_call(EXPERT_PROMPT,{"qid":q.qid,"question":q.question,"modality":mod},ev,q.qid,f"stage1_{mod}",MAMMQA_DO_SAMPLE,
                            lambda raw,s=f"stage1_{mod}":parse_tagged_agent(raw,s),False)
        parsed=envelope.get("parsed_output")
        if parsed:parsed["supporting_doc_ids"]=[x for x in parsed.get("supporting_doc_ids",[]) if x in allowed]
        usable=parse_stage1_output(envelope.get("raw_output"),mod); usable["supporting_doc_ids"]=[x for x in usable.get("supporting_doc_ids",[]) if x in allowed]
        envelope.update(schema_parse_success=envelope["parse_success"],usable_output=usable["usable"],usable_parsed_output=usable)
        out[mod]=envelope
    return out

In [19]:
# Cell 20 - compact paper-style anchor synthesis; no repair generation
SYNTH_PROMPT="""Synthesize from the named anchor expert and raw evidence from other modalities only.
Return exactly:
<insights>Maximum 2 concise sentences.</insights>
<answer>short candidate answer OR INSUFFICIENT</answer>
<subanswer>optional very short supporting result</subanswer>
<evidence_ids>comma-separated supplied IDs or NONE</evidence_ids>
No long explanation, chain-of-thought, or outside knowledge."""

def run_cross_modal_agents(q,experts,evidence):
    assert_inference_boundary("run_cross_modal_agents",q); out={}; allowed={str(d["parent_doc_id"]) for xs in evidence.values() for d in xs}
    for anchor,expert in experts.items():
        other=[]
        for mod in ["text","table","image"]:
            if mod!=anchor:other.extend(clean_evidence(evidence.get(mod,[]),mod,STAGE2_EVIDENCE_CHARS))
        anchor_payload=compact_or_raw(expert,1000)
        protected={"qid":q.qid,"question":q.question,"anchor_modality":anchor,"anchor_expert":anchor_payload}
        envelope=stage_call(SYNTH_PROMPT,protected,other,q.qid,f"stage2_{anchor}",MAMMQA_DO_SAMPLE,
                            lambda raw,s=f"stage2_{anchor}":parse_tagged_agent(raw,s),False)
        parsed=envelope.get("parsed_output")
        if parsed:parsed["supporting_doc_ids"]=[x for x in parsed.get("supporting_doc_ids",[]) if x in allowed]
        envelope["stage2_input_anchor_text"]=anchor_payload
        out[anchor]=envelope
    return out

In [20]:
# Cell 21 - one compact verifier over syntheses plus at most one evidence item/modality
VERIFY_PROMPT="""Classify whether the proposed answers are supported by the supplied evidence.
If supported, return exactly one line:
STATUS: supported
If information is missing or conflicts, return exactly three short lines:
STATUS: insufficient | conflict
RETRY_MODALITY: text | table | image
RETRY_QUERY: short targeted query
No answer, explanation, confidence, IDs, or other text."""

def verifier_evidence(synth,evidence):
    cited=set()
    for envelope in synth.values():
        parsed=envelope.get("parsed_output") if envelope.get("parse_success") else None
        if parsed:cited.update(map(str,parsed.get("supporting_doc_ids",[])))
    selected=[]
    for mod in ["text","table","image"]:
        docs=evidence.get(mod,[]); preferred=[d for d in docs if str(d["parent_doc_id"]) in cited]
        chosen=(preferred or docs)[:1]; selected.extend(clean_evidence(chosen,mod,VERIFIER_EVIDENCE_CHARS))
    return selected

def verify_answer(q,synth,evidence):
    assert_inference_boundary("verify_answer",q)
    protected={"qid":q.qid,"question":q.question,"candidate_syntheses":{m:compact_or_raw(x,900) for m,x in synth.items()}}
    envelope=stage_call(VERIFY_PROMPT,protected,verifier_evidence(synth,evidence),q.qid,"verifier",VERIFIER_DO_SAMPLE,parse_compact_verifier,False)
    envelope["verifier_input_candidate_syntheses"]=protected["candidate_syntheses"]
    parsed=envelope.get("parsed_output"); envelope["effective_status"]=parsed.get("status") if parsed else "verifier_error"; envelope["verifier_error"]=not bool(parsed); envelope["verifier_parse_failure"]=not bool(parsed)
    return envelope

In [21]:
# Cell 22 - one verifier pass, one optional feedback round, compact aggregation and compact traces
AGG_PROMPT="""Resolve the supplied synthesis results. Return exactly:
<answer>short final answer OR ABSTAIN</answer>
Optionally add <reason>one short sentence</reason>."""

def run_reasoning(q,evidence):
    t=time.perf_counter(); experts=run_modality_experts(q,evidence); stage1=time.perf_counter()-t
    t=time.perf_counter(); synth=run_cross_modal_agents(q,experts,evidence); stage2=time.perf_counter()-t
    return experts,synth,{"stage1_seconds":stage1,"stage2_seconds":stage2}

def aggregate_answer(q,synth,sees_question=True):
    protected={"qid":q.qid,"syntheses":{m:compact_or_raw(x,900) for m,x in synth.items()}}
    if sees_question:protected["question"]=q.question
    envelope=stage_call(AGG_PROMPT,protected,[],q.qid,"aggregator",MAMMQA_DO_SAMPLE,parse_compact_aggregator,True)
    envelope["aggregator_input_syntheses"]=protected["syntheses"]
    return envelope

def merge_evidence(old,new):
    out={}
    for mod in ["text","table","image"]:
        vals={d["parent_doc_id"]:d for d in old.get(mod,[])}
        for d in new.get(mod,[]):vals.setdefault(d["parent_doc_id"],d)
        out[mod]=list(vals.values())
    return out

def compact_pool(pool):
    keep=["parent_doc_id","doc_id","modality","final_score","dense_score","bm25_score","hybrid_rrf_score","graph_score","query_provenance"]
    return {m:[{k:d.get(k) for k in keep if d.get(k) is not None} for d in docs] for m,docs in pool.items()}

def compact_round(round_no,query_records,bundle):
    return {"round":round_no,"query_records":query_records,"candidate_ids":bundle["candidate_ids"],"context_ids":bundle["context_ids"],
            "routing":bundle["routing"],"retrieval_latency_seconds":bundle["retrieval_latency_seconds"]}

def feedback_request(verifier_parsed):
    if not isinstance(verifier_parsed,dict) or verifier_parsed.get("status") not in {"insufficient","conflict"}:return None
    mod=str(verifier_parsed.get("retry_modality","")).lower(); query=str(verifier_parsed.get("retry_query","")).strip()
    if mod not in {"text","table","image"} or not query:return None
    return {"retry_modality":mod,"retry_query":query,"retry_reason":verifier_parsed["status"]}

def execute_feedback_control(verifier_parsed,retrieve_one):
    request=feedback_request(verifier_parsed)
    if request is None:return None
    return {"request":request,"retrieval":retrieve_one(request)}

def run_pipeline(q,cfg):
    assert_inference_boundary("run_pipeline",q); started=time.perf_counter(); calls0=QWEN_CALL_COUNTER[q.qid]; repairs0=FORMAT_REPAIR_CALL_COUNTER[q.qid]
    t=time.perf_counter(); plan=fallback_plan(q.question) if not cfg["planner"] else plan_query(q); planner_seconds=time.perf_counter()-t
    bundle=retrieve_evidence(q,plan,retrieval_mode=cfg.get("retrieval_mode",RETRIEVAL_MODE),router_mode=cfg.get("router_mode",ROUTER_MODE),use_graph=cfg["graph"],fixed_all=not cfg["router"])
    evidence=bundle["context_evidence"]; experts,synth,reason_timing=run_reasoning(q,evidence)
    t=time.perf_counter(); verification=verify_answer(q,synth,evidence) if cfg["verifier"] else None; verifier_seconds=time.perf_counter()-t
    rounds=1; added=[]; targeted=None; feedback_seconds=0.0
    retrieval_round_details=[compact_round(1,plan["retrieval_queries"],bundle)]
    reasoning_round_details=[{"round":1,"experts":experts,"cross_modal":synth,"verification":verification}]
    verifier_parsed=(verification or {}).get("parsed_output") if (verification or {}).get("parse_success") else None
    def retrieve_one(request):
        mod=request["retry_modality"]; rq=request["retry_query"]; assert_clean_retrieval_query(rq)
        retry_plan={"retrieval_queries":build_query_records(q.question,[rq]),"normalized_subqueries":[rq],"needed_modalities":[mod],
                    "modality_confidence":{m:float(m==mod) for m in ["text","table","image"]},"routing_confidence":1.0}
        # Targeted feedback searches only the requested modality; graph expansion would violate that contract.
        retry=retrieve_evidence(q,retry_plan,retrieval_mode=cfg.get("retrieval_mode",RETRIEVAL_MODE),router_mode="hard",use_graph=False)
        return {"plan":retry_plan,"bundle":retry}
    feedback_start=time.perf_counter(); dispatch=execute_feedback_control(verifier_parsed,retrieve_one) if cfg["feedback"] else None
    if dispatch:
        request=dispatch["request"]; mod=request["retry_modality"]; rq=request["retry_query"]
        retry_plan=dispatch["retrieval"]["plan"]; retry=dispatch["retrieval"]["bundle"]
        before={d["parent_doc_id"] for xs in evidence.values() for d in xs}; evidence=merge_evidence(evidence,retry["context_evidence"])
        bundle["context_evidence"]=evidence; bundle["retrieval_candidates"]=merge_evidence(bundle["retrieval_candidates"],retry["retrieval_candidates"])
        bundle["candidate_ids"]={m:[d["parent_doc_id"] for d in xs] for m,xs in bundle["retrieval_candidates"].items()}; bundle["context_ids"]={m:[d["parent_doc_id"] for d in xs] for m,xs in evidence.items()}
        after={d["parent_doc_id"] for xs in evidence.values() for d in xs}; added=sorted(after-before); rounds=2
        targeted={"retry_triggered":True,"retry_reason":request["retry_reason"],"retry_modality":mod,"retry_query":rq,"feedback_round_count":1,"new_candidate_ids":retry["candidate_ids"],"new_context_ids":retry["context_ids"]}
        retrieval_round_details.append(compact_round(2,retry_plan["retrieval_queries"],retry))
        experts,synth,retry_timing=run_reasoning(q,evidence)  # deliberately no second verifier
        reason_timing["stage1_seconds"]+=retry_timing["stage1_seconds"]; reason_timing["stage2_seconds"]+=retry_timing["stage2_seconds"]
        reasoning_round_details.append({"round":2,"experts":experts,"cross_modal":synth,"verification":None,"second_verifier_skipped":True})
        feedback_seconds=time.perf_counter()-feedback_start
    t=time.perf_counter(); agg=aggregate_answer(q,synth,cfg["agg_question"]); aggregator_seconds=time.perf_counter()-t
    answer,answer_ok=parse_final_answer(agg); agg_parsed=agg.get("parsed_output") if agg.get("parse_success") else None
    final_status=agg_parsed.get("status") if agg_parsed else "generation_error"; forced=False
    if agg_parsed and agg_parsed.get("status")=="abstain":answer="ABSTAIN"; final_status="abstain"; answer_ok=True
    timing={"planner_seconds":planner_seconds,"retrieval_seconds":sum(x["retrieval_latency_seconds"] for x in retrieval_round_details),
            **reason_timing,"verifier_seconds":verifier_seconds,"feedback_seconds":feedback_seconds,"aggregator_seconds":aggregator_seconds,
            "total_seconds":time.perf_counter()-started}
    final={"answer":answer,"status":final_status,"forced_by_valid_verifier":forced,"answer_parse_success":answer_ok,"infrastructure_failure":not answer_ok}
    information_trace={"qid":q.qid,"stage1_raw_outputs":{m:x.get("raw_output","") for m,x in experts.items()},
        "stage1_schema_parse":{m:bool(x.get("schema_parse_success")) for m,x in experts.items()},
        "stage1_usable":{m:bool(x.get("usable_output")) for m,x in experts.items()},
        "stage2_input_anchor_text":{m:x.get("stage2_input_anchor_text") for m,x in synth.items()},
        "stage2_outputs":{m:x.get("raw_output","") for m,x in synth.items()},
        "verifier_input_candidate_syntheses":(verification or {}).get("verifier_input_candidate_syntheses",{}),
        "verifier_status":(verifier_parsed or {}).get("status","verifier_error"),
        "aggregator_input_syntheses":agg.get("aggregator_input_syntheses",{}),"final_answer":answer}
    append_jsonl_log(INFORMATION_TRACE_PATH,information_trace)
    return {"qid":q.qid,"question":q.question,"inference_input_fields":["qid","question"],"planner":plan,
            "retrieval_candidates":compact_pool(bundle["retrieval_candidates"]),"context_evidence":compact_pool(evidence),"retrieved":compact_pool(evidence),
            "candidate_ids":bundle["candidate_ids"],"context_ids":bundle["context_ids"],"routing":bundle["routing"],
            "retrieval_latency_seconds":bundle["retrieval_latency_seconds"],"graph_provenance_count":len(bundle.get("graph_provenance",[])),
            "verification":verification,"aggregator_raw":agg,"raw_final_output":agg.get("raw_output"),
            "parsed_final_answer":agg.get("parsed_output"),"final_decision":final,"retrieval_round_details":retrieval_round_details,
            "reasoning_round_details":reasoning_round_details,"information_trace":information_trace,"targeted_feedback":targeted,"retrieval_rounds":rounds,"new_docs_added":added,
            "final_answer":answer,"abstain":answer=="ABSTAIN","forced_parser_abstain":False,"timing":timing,
            "qwen_call_count":QWEN_CALL_COUNTER[q.qid]-calls0,"format_repair_call_count":FORMAT_REPAIR_CALL_COUNTER[q.qid]-repairs0,
            "latency_seconds":timing["total_seconds"]}

In [22]:
# Cell 23 - exact controls; MAMMQA_ORACLE is a pure reasoning upper bound
CONFIGS={
 "B1_Dense":{"planner":False,"router":False,"router_mode":"all","retrieval_mode":"dense","graph":False,"verifier":False,"feedback":False,"agg_question":True},
 "B2_Planner_Dense":{"planner":True,"router":True,"router_mode":"soft","retrieval_mode":"dense","graph":False,"verifier":False,"feedback":False,"agg_question":True},
 "FULL":{"planner":True,"router":True,"router_mode":"soft","retrieval_mode":"hybrid","graph":True,"verifier":True,"feedback":True,"agg_question":True},
}

def oracle_evidence(ev):
    evidence={m:[] for m in ["text","table","image"]}
    for did in ev.gold_support_ids:
        mod=ev.gold_support_modalities.get(did)
        recs=PARENT_LOOKUP.get((mod,did),[]) if mod else []
        for rec in recs:
            if rec and (rec["modality"]!="image" or rec.get("image_path")):
                evidence[rec["modality"]].append({**rec,"dense_score":1.0,"bm25_score":1.0,"final_score":1.0,"query_provenance":[{"query_text":"ORACLE_GOLD_EVIDENCE","query_source":"oracle","rank":1,"score":1.0,"method":"oracle"}]})
    return evidence

def oracle_pipeline(q,ev):
    assert_inference_boundary("oracle_pipeline",q); started=time.perf_counter(); evidence=oracle_evidence(ev)
    experts,synth,_=run_reasoning(q,evidence); agg=aggregate_answer(q,synth,True); answer,ok=parse_final_answer(agg)
    parsed=agg.get("parsed_output") if agg.get("parse_success") else None
    if parsed and parsed.get("status")=="abstain":answer="ABSTAIN"; ok=True
    final={"answer":answer,"status":parsed.get("status") if parsed else "generation_error","confidence":clip01((parsed or {}).get("confidence")),"answer_parse_success":ok,"infrastructure_failure":not ok}
    return {"qid":q.qid,"question":q.question,"inference_input_fields":["qid","question"],"planner":{"oracle":True},
            "retrieval_candidates":{},"context_evidence":evidence,"retrieved":evidence,"candidate_ids":{},"context_ids":{m:[d["parent_doc_id"] for d in xs] for m,xs in evidence.items()},
            "stage1":experts,"cross_modal":synth,"verification":None,"aggregator_raw":agg,"raw_final_output":agg.get("raw_output"),
            "parsed_final_answer":agg.get("parsed_output"),"final_decision":final,"retrieval_round_details":[],
            "reasoning_round_details":[{"round":1,"experts":experts,"cross_modal":synth,"verification":None}],"targeted_feedback":None,
            "retrieval_rounds":0,"new_docs_added":[],"final_answer":answer,"confidence":final["confidence"],"abstain":answer=="ABSTAIN",
            "forced_parser_abstain":False,"latency_seconds":time.perf_counter()-started}

In [23]:
# Cell 24 - gold parsing and candidate/context retrieval metrics
def answer_strings(row):
    vals=[]
    def walk(x):
        if isinstance(x,str):vals.append(x)
        elif isinstance(x,list):
            for y in x:walk(y)
        elif isinstance(x,dict):
            for k in ["answer","answers","aliases","normalized_answer"]:
                if k in x:walk(x[k])
    walk(row.get("answers",[])); return list(dict.fromkeys(v.strip() for v in vals if v.strip()))

def support_info(row):
    ids=[]; mods={}; known={m:{str(x["id"]) for x in raw} for m,raw in [("text",texts_raw),("table",tables_raw),("image",images_raw)]}
    def add(v,mod=None):
        s=str(v); inferred=mod or next((m for m,z in known.items() if s in z),None)
        if s not in {"","None"} and inferred and s in known[inferred]:ids.append(s); mods[s]=inferred
    def walk(x,mod=None):
        if isinstance(x,list):
            for y in x:walk(y,mod)
        elif isinstance(x,dict):
            local={"text":"text","table":"table","image":"image"}.get(str(x.get("modality",x.get("doc_part",mod))).lower(),mod)
            for k,v in x.items():
                km={"text_doc_id":"text","table_id":"table","image_doc_id":"image","doc_id":local}.get(k)
                if km:add(v,km)
                elif k not in {"answer","text","content"}:walk(v,local)
        elif mod:add(x,mod)
    walk(row.get("supporting_context",[])); return list(dict.fromkeys(ids)),mods

def evaluation_only(row):
    ids,mods=support_info(row); meta=row.get("metadata",{})
    return EvaluationOnly(answer_strings(row),ids,mods,str(meta.get("type","unknown")),list(meta.get("modalities",[])),row.get("intermediate_answers"))

ARTICLES={"a","an","the"}
def norm_answer(s):
    s=str(s).lower(); s="".join(" " if c in string.punctuation else c for c in s); return " ".join(w for w in s.split() if w not in ARTICLES)
def em_f1(pred,golds):
    p=norm_answer(pred).split(); best_em=0; best_f1=0.0
    for g0 in golds:
        g=norm_answer(g0).split(); best_em=max(best_em,int(p==g)); common=Counter(p)&Counter(g); n=sum(common.values())
        score=0 if not p or not g or not n else 2*(n/len(p))*(n/len(g))/((n/len(p))+(n/len(g)))
        best_f1=max(best_f1,score)
    return best_em,best_f1

def ranked_parent_ids(result,k,mod=None,pool="retrieval_candidates"):
    pools=result.get(pool,{}) or {}; docs=pools.get(mod,[]) if mod else [x for xs in pools.values() for x in xs]
    docs=sorted(docs,key=lambda x:x.get("final_score",0),reverse=True)
    return list(dict.fromkeys(str(x["parent_doc_id"]) for x in docs))[:k]

def retrieval_scores(result,ev):
    out={}; gold=set(map(str,ev.gold_support_ids))
    for k in [1,3,5,10]:
        ids=ranked_parent_ids(result,k); hit=len(gold&set(ids)); out[f"recall@{k}"]=hit/len(gold) if gold else math.nan
        out[f"precision@{k}"]=hit/len(ids) if ids else 0.0; out[f"hit@{k}"]=int(hit>0) if gold else math.nan
        out[f"all_support_recall@{k}"]=int(gold<=set(ids)) if gold else math.nan
        for mod in ["text","table","image"]:
            gm={x for x in gold if ev.gold_support_modalities.get(x)==mod}; mids=set(ranked_parent_ids(result,k,mod))
            out[f"{mod}_recall@{k}"]=len(gm&mids)/len(gm) if gm else math.nan
    ranked=ranked_parent_ids(result,10**9); out["mrr"]=next((1/(i+1) for i,x in enumerate(ranked) if x in gold),0.0) if gold else math.nan
    candidate=set(ranked); context=set(ranked_parent_ids(result,10**9,pool="context_evidence"))
    out["gold_candidate_pool_coverage"]=len(gold&candidate)/len(gold) if gold else math.nan
    out["gold_context_coverage"]=len(gold&context)/len(gold) if gold else math.nan
    out["gold_in_candidate_pool"]=int(gold<=candidate) if gold else math.nan; out["gold_in_context"]=int(gold<=context) if gold else math.nan
    out["retrieved_document_count"]=len(candidate); out["context_document_count"]=len(context)
    gold_mods=set(ev.gold_support_modalities.values()); selected=set(result.get("planner",{}).get("needed_modalities",[]))
    out["planner_modality_recall"]=len(gold_mods&selected)/len(gold_mods) if gold_mods else math.nan
    return out

In [24]:
# Cell 25 - exact frozen smoke sample, fast-validation gate and optional deterministic sharding
FROZEN_QIDS=[
 "f7dd1ac143507bd2ab9a480f18b86706","254afe636a34c3a5d94efa22d95465ad",
 "ebb69554283456251305b4b8ab579f76","4b9a4868455f98a69a8494ef1dfe7205",
 "186146ea1f5a9f7ada7f7fd31c5c75a4","c9fcae0ac6e3aa3f08375db9e4fe9037",
 "5dfd4afc175cdebf43295316a540d70c","6d372c501004c3e448b1991f39160e99",
 "b5d7685e595423638d26dd60e99fcc39","3126664c6ce707c6aca2c475ace8db08",
 "183a1eec0e38eaee513a5ec7996f55f1","2a5fdab4235e28037389777f473c4f9f",
 "2f154508615dc2d34b4d4b7fdced0b0e","22e22f3b558eeae2178ddd9cb6bc2b04",
 "1e3f1a08d865b3676a5ca81d8adc270b","ceb2b6a63d5f3b5eb8075979a55e7471"]
row_by_qid={query_only(r).qid:r for r in dev_raw}; missing=[qid for qid in FROZEN_QIDS if qid not in row_by_qid]
assert not missing,f"Frozen qids missing: {missing}"
if RUN_MODE=="full":SELECTED_BASE=list(dev_raw)
else:SELECTED_BASE=[row_by_qid[qid] for qid in FROZEN_QIDS]
if FAST_VALIDATION:
    assert RUN_MODE=="smoke" and NUM_SHARDS==1 and SHARD_ID==0
    SELECTED_BASE=SELECTED_BASE[:2]
SELECTED=SELECTED_BASE[SHARD_ID::NUM_SHARDS]; SELECTED_QIDS=[query_only(r).qid for r in SELECTED]
assert len(SELECTED_QIDS)==len(set(SELECTED_QIDS)) and SELECTED_QIDS
if FAST_VALIDATION:assert SELECTED_QIDS==FROZEN_QIDS[:2]
elif RUN_MODE=="smoke":assert SELECTED_QIDS==FROZEN_QIDS[SHARD_ID::NUM_SHARDS]
(OUTPUT_ROOT/"selected_questions.json").write_text(json.dumps([asdict(query_only(r)) for r in SELECTED],indent=2,ensure_ascii=False),encoding="utf-8")
print("FAST_VALIDATION =",FAST_VALIDATION,"| NUM_SHARDS =",NUM_SHARDS,"| SHARD_ID =",SHARD_ID)
print("Selected qids:",SELECTED_QIDS)

FAST_VALIDATION = False | NUM_SHARDS = 1 | SHARD_ID = 0
Selected qids: ['f7dd1ac143507bd2ab9a480f18b86706', '254afe636a34c3a5d94efa22d95465ad', 'ebb69554283456251305b4b8ab579f76', '4b9a4868455f98a69a8494ef1dfe7205', '186146ea1f5a9f7ada7f7fd31c5c75a4', 'c9fcae0ac6e3aa3f08375db9e4fe9037', '5dfd4afc175cdebf43295316a540d70c', '6d372c501004c3e448b1991f39160e99', 'b5d7685e595423638d26dd60e99fcc39', '3126664c6ce707c6aca2c475ace8db08', '183a1eec0e38eaee513a5ec7996f55f1', '2a5fdab4235e28037389777f473c4f9f', '2f154508615dc2d34b4d4b7fdced0b0e', '22e22f3b558eeae2178ddd9cb6bc2b04', '1e3f1a08d865b3676a5ca81d8adc270b', 'ceb2b6a63d5f3b5eb8075979a55e7471']


In [25]:
# Cell 26 - pre-run acceptance checks and requested before/after engineering table
assert build_query_records("Original question?",[{"query":"focused query"},"second query"])[0]=={"text":"Original question?","source":"original_question"}
assert parse_compact_planner("MODALITIES: text,table,image\nWEIGHTS: text=0.4,table=0.3,image=0.3\nQUERY1: focused")
assert parse_tagged_agent("MODALITY: text\nANSWER: candidate\nINSIGHT: short\nSUBANSWER: NONE","stage1_text")
assert parse_tagged_agent("MODALITY: image\nANSWER: candidate\nINSIGHT: short","stage1_image")["subanswer"]=="NONE"
_free_stage1=parse_stage1_output("useful unlabeled natural-language evidence","image")
assert _free_stage1["usable"] and not _free_stage1["schema_parse_success"] and _free_stage1["candidate_answer"] is None
assert _free_stage1["insights"]=="useful unlabeled natural-language evidence"
assert not parse_stage1_output("   ","text")["usable"]
for variant in ["STATUS: supported","supported","STATUS=supported"]:
    parsed=parse_compact_verifier(variant); assert parsed and parsed["status"]=="supported" and not parsed["feedback_possible"]
for variant in ["STATUS: insufficient","insufficient","STATUS=conflict"]:assert parse_compact_verifier(variant)
assert parse_compact_verifier("nonsense") is None
assert parse_compact_verifier("STATUS: supported\nunrelated text") is None
_insufficient=parse_compact_verifier("STATUS: insufficient\nRETRY_MODALITY: image\nRETRY_QUERY: What color is the logo?")
_conflict=parse_compact_verifier("STATUS: conflict\nRETRY_MODALITY: text\nRETRY_QUERY: Find the release date.")
assert feedback_request(parse_compact_verifier("STATUS: supported")) is None
assert feedback_request(_insufficient)=={"retry_modality":"image","retry_query":"What color is the logo?","retry_reason":"insufficient"}
assert feedback_request(_conflict)=={"retry_modality":"text","retry_query":"Find the release date.","retry_reason":"conflict"}
assert feedback_request(parse_compact_verifier("STATUS: insufficient")) is None
_feedback_calls=[]
def _fake_targeted_retrieval(request):
    _feedback_calls.append(request); return {"candidate_modalities":[request["retry_modality"]]}
_dispatch=execute_feedback_control(_insufficient,_fake_targeted_retrieval)
assert len(_feedback_calls)==1 and _feedback_calls[0]["retry_modality"]=="image"
assert _dispatch["retrieval"]["candidate_modalities"]==["image"]
assert execute_feedback_control(parse_compact_verifier("STATUS: supported"),_fake_targeted_retrieval) is None
assert len(_feedback_calls)==1
FEEDBACK_UNIT_TESTS_PASS=True
assert parse_compact_aggregator("<answer>yes</answer>")
candidate_budget,context_budget=route_budgets({"needed_modalities":["text"],"modality_confidence":{"text":.8,"table":.1,"image":.1},"routing_confidence":.8},"soft")
available=[m for m in ["text","table","image"] if BY_MODALITY[m] and (m!="image" or IMAGE_PATHS)]
assert set(candidate_budget)==set(available) and all(candidate_budget[m]>=20 for m in available)
assert (TEXT_CONTEXT_K,TABLE_CONTEXT_K,IMAGE_CONTEXT_K)==(3,2,2)

OPTIMIZATION_ROWS=[
 ["Methods run","FULL only","FULL only"],["Retrieval candidate K","20","20 (unchanged)"],
 ["Text context K","5","3"],["Table context K","3","2"],["Image context K","3","2"],
 ["Planner output","verbose JSON","compact lines"],["Planner repair","yes","no; deterministic fallback"],
 ["Stage-I output","verbose JSON","compact lines + nonempty raw usability fallback"],["Stage-I repair","yes","no"],
 ["Stage-II output","verbose JSON","compact tags"],["Stage-II repair","yes","no"],
 ["Verifier output","verbose JSON","conditional 1/3-line status protocol"],["Verifier repair","yes","no"],
 ["Verifier evidence","all context","top/cited 1 per modality"],["Aggregator repair","yes","one maximum"],
 ["Stage token limits","128/160/192/128/96","64/(96,96,128)/128/48/48"],
 ["gc.collect per call","yes","no; once per 4 questions"],["empty_cache per call","yes","OOM recovery only"],
 ["BGE device","CPU after indexing",BGE_DEVICE],["CLIP device","CPU",CLIP_DEVICE],
 ["Persistent indexes","working cache only","loadable input cache + publishable package"],
 ["Image manifest","recursive scan","persistent manifest with 57,058 entries"],
 ["Checkpoint/resume","yes","yes; fsync each question"],["Sharding","no",f"yes ({SHARD_ID}/{NUM_SHARDS})"],
 ["Fast validation","no",str(FAST_VALIDATION)]]
OPTIMIZATION_DF=pd.DataFrame(OPTIMIZATION_ROWS,columns=["Component","Before","After"])
print("\nRUNTIME OPTIMIZATION SUMMARY\n",OPTIMIZATION_DF.to_string(index=False))
OPTIMIZATION_SUMMARY_PATH.write_text(json.dumps({"changes":[dict(zip(["component","before","after"],r)) for r in OPTIMIZATION_ROWS],"ready_for_16":None},indent=2),encoding="utf-8")
print("FEEDBACK CONTROL UNIT TESTS: PASS")
print("PRE-RUN ACCEPTANCE PASSED. This run is limited to",len(SELECTED),"question(s).")


RUNTIME OPTIMIZATION SUMMARY
             Component             Before                                           After
          Methods run          FULL only                                       FULL only
Retrieval candidate K                 20                                  20 (unchanged)
       Text context K                  5                                               3
      Table context K                  3                                               2
      Image context K                  3                                               2
       Planner output       verbose JSON                                   compact lines
       Planner repair                yes                      no; deterministic fallback
       Stage-I output       verbose JSON compact lines + nonempty raw usability fallback
       Stage-I repair                yes                                              no
      Stage-II output       verbose JSON                                    com

In [26]:
# Cell 27 - FULL-only checkpointed inference; gold remains unavailable until every selected prediction is frozen
def append_jsonl(path,row):append_jsonl_log(path,row)
def read_jsonl_checked(path):
    path=Path(path)
    if not path.exists():return []
    rows=[]
    with path.open(encoding="utf-8") as f:
        for no,line in enumerate(f,1):
            if line.strip():
                try:rows.append(json.loads(line))
                except json.JSONDecodeError as e:raise ValueError(f"{path}:{no}: {e}") from e
    return rows

def clean_for_json(x):
    if isinstance(x,Path):return str(x)
    if isinstance(x,dict):return {k:clean_for_json(v) for k,v in x.items() if k!="source_record"}
    if isinstance(x,(list,tuple)):return [clean_for_json(v) for v in x]
    if isinstance(x,np.generic):return x.item()
    return x

def completed(path):
    rows=read_jsonl_checked(path); qids=[r["qid"] for r in rows]
    assert len(qids)==len(set(qids)); assert qids==SELECTED_QIDS[:len(qids)],f"Stale or out-of-order checkpoint: {qids}"
    return set(qids)

CONFIGS={"FULL":{"planner":True,"router":True,"router_mode":"soft","retrieval_mode":"hybrid","graph":True,"verifier":True,"feedback":True,"agg_question":True}}
METHODS=["FULL"]
prediction_name="FULL.jsonl" if NUM_SHARDS==1 else f"FULL_shard_{SHARD_ID}.jsonl"
IMPLEMENTATION_FINGERPRINT=hashlib.sha256(json.dumps({"prompts":[PLANNER_PROMPT,EXPERT_PROMPT,SYNTH_PROMPT,VERIFY_PROMPT,AGG_PROMPT],"stage_tokens":STAGE_MAX_NEW_TOKENS,"context":[TEXT_CONTEXT_K,TABLE_CONTEXT_K,IMAGE_CONTEXT_K],"candidate_k":RETRIEVAL_CANDIDATE_K,"stage1_nonempty_raw_is_usable":True},sort_keys=True).encode()).hexdigest()
EXPERIMENT_CONFIG={"experiment_id":EXPERIMENT_ID,"implementation_fingerprint":IMPLEMENTATION_FINGERPRINT,"model":MODEL_NAME,"seed":SEED,
 "temperature":MAMMQA_TEMPERATURE,"top_p":MAMMQA_TOP_P,"do_sample":MAMMQA_DO_SAMPLE,"stage_token_limits":STAGE_MAX_NEW_TOKENS,
 "max_input_tokens":MAX_INPUT_TOKENS,"context_k":{"text":TEXT_CONTEXT_K,"table":TABLE_CONTEXT_K,"image":IMAGE_CONTEXT_K},
 "fallback_context_k":{"text":FALLBACK_TEXT_K,"table":FALLBACK_TABLE_K,"image":FALLBACK_IMAGE_K},"candidate_k":RETRIEVAL_CANDIDATE_K,
 "retrieval_mode":RETRIEVAL_MODE,"router_mode":ROUTER_MODE,"graph":{"enabled":ENABLE_GRAPH,"seed_k":GRAPH_SEED_K,"max_expanded":GRAPH_MAX_EXPANDED_DOCS,"max_degree":GRAPH_MAX_DEGREE},
 "verifier":True,"feedback":True,"max_retrieval_rounds":MAX_RETRIEVAL_ROUNDS,"image_processor_use_fast":IMAGE_PROCESSOR_USE_FAST,
 "bge_device":BGE_DEVICE,"clip_device_requested":CLIP_DEVICE,"fast_validation":FAST_VALIDATION,"num_shards":NUM_SHARDS,"shard_id":SHARD_ID,
 "qids":SELECTED_QIDS,"methods":METHODS,"web_enabled":ENABLE_WEB,"persistent_cache_input":str(PERSISTENT_CACHE_INPUT)}
if RUNTIME_CONFIG_PATH.exists():
    prior_config=json.loads(RUNTIME_CONFIG_PATH.read_text())
    mutable_selection_fields={"fast_validation","qids"}
    assert {k:v for k,v in prior_config.items() if k not in mutable_selection_fields}=={k:v for k,v in EXPERIMENT_CONFIG.items() if k not in mutable_selection_fields},"Experiment id has incompatible inference settings"
RUNTIME_CONFIG_PATH.write_text(json.dumps(EXPERIMENT_CONFIG,indent=2),encoding="utf-8")

STARTUP_SECONDS=time.perf_counter()-NOTEBOOK_START_TIME; STARTUP_TIMINGS["total_startup_seconds"]=STARTUP_SECONDS
print("STARTUP TIMING",json.dumps(STARTUP_TIMINGS,indent=2))
print("CACHE RESOURCES",json.dumps(CACHE_RESOURCE_STATUS,indent=2))
def print_stage1_health(result):
    experts=(result.get("reasoning_round_details") or [{}])[-1].get("experts",{})
    print("STAGE-I HEALTH")
    usable=0
    for mod in ["text","table","image"]:
        env=experts.get(mod,{})
        schema=bool(env.get("schema_parse_success",env.get("parse_success"))); ok=bool(env.get("usable_output")); usable+=int(ok)
        print(f"{mod}: schema_parse = {schema} usable = {ok}")
    print(f"overall usable: {usable}/{len(experts)}")

def run_full(rows):
    path=DIRS["predictions"]/prediction_name; existing=read_jsonl_checked(path); done=completed(path); completed_times=[float(r.get("latency_seconds",0)) for r in existing]
    timed={r.get("qid") for r in read_jsonl_checked(TIMING_LOG_PATH)}
    for row in existing:
        if row["qid"] not in timed:append_jsonl(TIMING_LOG_PATH,{"qid":row["qid"],**row.get("timing",{}),"qwen_call_count":row.get("qwen_call_count"),"format_repair_call_count":row.get("format_repair_call_count")})
    print(f"[FULL] {len(done)}/{len(rows)} already complete | startup_seconds={STARTUP_SECONDS:.1f}",flush=True)
    for position,raw in enumerate(rows,1):
        q=query_only(raw)
        if q.qid in done:print(f"[FULL] {position}/{len(rows)} RESUME-SKIP {q.qid}",flush=True); continue
        started=time.perf_counter(); result=clean_for_json(run_pipeline(q,CONFIGS["FULL"])); print_stage1_health(result); append_jsonl(path,result); done.add(q.qid)
        elapsed=time.perf_counter()-started; completed_times.append(elapsed); append_jsonl(TIMING_LOG_PATH,{"qid":q.qid,**result["timing"],"qwen_call_count":result["qwen_call_count"],"format_repair_call_count":result["format_repair_call_count"]})
        complete=len(done); avg=float(np.mean(completed_times)); remaining=max(0,len(rows)-complete)*avg/60
        print(f"[FULL] {complete}/{len(rows)} complete | time_this_question={elapsed:.1f}s | average_question_time={avg:.1f}s | estimated_remaining_minutes={remaining:.1f} | qwen_calls={result['qwen_call_count']} | repair_calls={result['format_repair_call_count']}",flush=True)
        if complete%4==0:gc.collect()
    return path

assert model is not None and processor is not None and METHODS==["FULL"]
INFERENCE_STARTED=time.perf_counter(); RESULT_PATHS={"FULL":run_full(SELECTED)}; INFERENCE_SECONDS=time.perf_counter()-INFERENCE_STARTED
PREDICTION_ROWS={"FULL":read_jsonl_checked(RESULT_PATHS["FULL"])}
assert [r["qid"] for r in PREDICTION_ROWS["FULL"]]==SELECTED_QIDS
assert all(set(r.get("inference_input_fields",[]))=={"qid","question"} for r in PREDICTION_ROWS["FULL"])
print("\nPREDICTIONS FROZEN. Gold evaluation begins only now.")
for r in PREDICTION_ROWS["FULL"]:print(r["qid"],"=>",r["final_answer"])

STARTUP TIMING {
  "image_manifest_seconds": 82.72449223800004,
  "retrieval_cache_seconds": 685.3730902240002,
  "qwen_load_seconds": 122.77263266900013,
  "total_startup_seconds": 975.202536042
}
CACHE RESOURCES {
  "image_manifest": "BUILT",
  "text": "BUILT",
  "table": "BUILT",
  "image_titles": "BUILT",
  "text_bm25": "BUILT",
  "table_bm25": "BUILT",
  "image_bm25": "BUILT"
}
[FULL] 0/16 already complete | startup_seconds=975.2
[QWEN START] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=planner input=168 images=0 max_new=64
[QWEN DONE] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=planner output=27 seconds=2.9
[PARSE] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=planner success=True repair=False


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (5 outputs left out)

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP actual device = cuda:0
[QWEN START] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=stage1_text input=345 images=0 max_new=96
[QWEN DONE] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=stage1_text output=29 seconds=2.8
[PARSE] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=stage1_text success=False repair=False
[QWEN START] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=stage1_table input=470 images=0 max_new=96
[QWEN DONE] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=stage1_table output=22 seconds=2.3
[PARSE] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=stage1_table success=False repair=False
[QWEN START] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=stage1_image input=840 images=2 max_new=128
[QWEN DONE] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=stage1_image output=8 seconds=3.6
[PARSE] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=stage1_image success=False repair=False
[QWEN START] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=stage2_text input=1231 images=2 max_new=128
[QWEN DONE] qid=f7dd1ac143507bd2ab9

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


[QWEN DONE] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=verifier output=4 seconds=2.3
[PARSE] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=verifier success=True repair=False
[QWEN START] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=aggregator input=494 images=0 max_new=48
[QWEN DONE] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=aggregator output=10 seconds=1.3
[PARSE] qid=f7dd1ac143507bd2ab9a480f18b86706 stage=aggregator success=True repair=False
STAGE-I HEALTH
text: schema_parse = False usable = True
table: schema_parse = False usable = True
image: schema_parse = False usable = True
overall usable: 3/3
[FULL] 1/16 complete | time_this_question=60.9s | average_question_time=60.9s | estimated_remaining_minutes=15.2 | qwen_calls=9 | repair_calls=0
[QWEN START] qid=254afe636a34c3a5d94efa22d95465ad stage=planner input=181 images=0 max_new=64
[QWEN DONE] qid=254afe636a34c3a5d94efa22d95465ad stage=planner output=46 seconds=3.7
[PARSE] qid=254afe636a34c3a5d94efa22d95465ad stage=planner success=True 

In [27]:
# Cell 28 - post-freeze evaluation, two-question runtime acceptance and mandatory stop
def mean_valid(vals):
    vals=[float(v) for v in vals if v is not None and not (isinstance(v,float) and math.isnan(v))]; return float(np.mean(vals)) if vals else math.nan
def envelopes_for(row,kind):
    if kind=="planner":return [row.get("planner",{}).get("planner_generation")]
    if kind=="stage1":return [x for rr in row.get("reasoning_round_details",[]) for x in rr.get("experts",{}).values()]
    if kind=="stage2":return [x for rr in row.get("reasoning_round_details",[]) for x in rr.get("cross_modal",{}).values()]
    if kind=="verifier":return [rr.get("verification") for rr in row.get("reasoning_round_details",[]) if rr.get("verification") is not None]
    if kind=="aggregator":return [row.get("aggregator_raw")]
    return []
def stage_health(rows,kind):
    env=[x for r in rows for x in envelopes_for(r,kind) if x]; return {"parsed":sum(bool(x.get("parse_success")) for x in env),"total":len(env),"rate":sum(bool(x.get("parse_success")) for x in env)/len(env) if env else 0.0}
def stage1_usable_health(rows):
    env=[x for r in rows for x in envelopes_for(r,"stage1") if x]; usable=sum(bool(x.get("usable_output")) for x in env)
    return {"usable":usable,"total":len(env),"rate":usable/len(env) if env else 0.0}
def modality_health(rows,kind,mod,predicate):
    env=[]
    for r in rows:
        for rr in r.get("reasoning_round_details",[]):
            group=rr.get("experts",{}) if kind=="stage1" else rr.get("cross_modal",{})
            if group.get(mod):env.append(group[mod])
    passed=sum(bool(predicate(x)) for x in env); return {"passed":passed,"total":len(env),"rate":passed/len(env) if env else 0.0}

# Gold is materialized only after all selected predictions are frozen above.
evaluated=[]
for prediction in PREDICTION_ROWS["FULL"]:
    raw=row_by_qid[prediction["qid"]]; ev=evaluation_only(raw); row=copy.deepcopy(prediction); em,f1=em_f1(row["final_answer"],ev.gold_answers)
    row.update(gold_answers=ev.gold_answers,gold_support_ids=ev.gold_support_ids,gold_support_modalities=ev.gold_support_modalities,
               exact_match=em,token_f1=f1,retrieval_metrics=retrieval_scores(row,ev)); evaluated.append(row)
FULL_ROWS=evaluated

def all_parent_ids(row,pool):return {str(d["parent_doc_id"]) for docs in (row.get(pool,{}) or {}).values() for d in docs}
def parsed_candidate_answers(row,kind):
    out=[]
    for envelope in envelopes_for(row,kind):
        parsed=(envelope or {}).get("parsed_output") or {}; value=parsed.get("candidate_answer",parsed.get("answer",""))
        if value:out.append(value)
    return out
def matches_gold(value,row):return em_f1(str(value or ""),row.get("gold_answers",[]))[0]==1
def classify_failure(row):
    if row["exact_match"]:return "correct"
    planner=row.get("planner",{}); planner_env=planner.get("planner_generation") or {}
    if planner.get("planner_parse_failure"):
        return "planner_generation_failure" if not planner_env.get("raw_output") else "planner_parse_failure"
    for rr in row.get("reasoning_round_details",[]):
        for mod,env in rr.get("experts",{}).items():
            if not env.get("usable_output"):return f"stage1_{mod}_failure"
        if any(env.get("generation_schema_failure") for env in rr.get("cross_modal",{}).values()):return "stage2_generation_failure"
    verifier=row.get("verification") or {}; vp=verifier.get("parsed_output") or {}
    if verifier.get("verifier_error"):
        return "verifier_parse_failure" if verifier.get("raw_output") else "verifier_generation_failure"
    if any(x.get("generation_schema_failure") for x in envelopes_for(row,"aggregator") if x):return "aggregation_generation_failure"
    if not row.get("final_decision",{}).get("answer_parse_success"):return "answer_parser_failure"
    gold=set(map(str,row.get("gold_support_ids",[]))); candidates=all_parent_ids(row,"retrieval_candidates"); context=all_parent_ids(row,"context_evidence")
    if gold and not gold<=candidates:return "retrieval_failure"
    if gold and not gold<=context:return "context_selection_failure"
    if row.get("retrieval_rounds")==2 and gold and not (gold&set(map(str,row.get("new_docs_added",[])))):return "feedback_retrieval_failure"
    if vp.get("status")=="insufficient":
        return "abstention_due_to_insufficient_evidence" if row.get("abstain") else "verification_insufficient"
    if vp.get("status")=="conflict":return "verification_conflict"
    if any(matches_gold(x,row) for x in parsed_candidate_answers(row,"stage2")):return "aggregation_failure"
    return "aggregation_failure"

failure_rows=[{"qid":r["qid"],"failure_type":classify_failure(r),"candidate_gold_coverage":r["retrieval_metrics"].get("gold_candidate_pool_coverage"),
               "context_gold_coverage":r["retrieval_metrics"].get("gold_context_coverage"),"retry_used":r.get("retrieval_rounds")==2} for r in FULL_ROWS]
pd.DataFrame(failure_rows).to_csv(DIRS["metrics"]/"failure_analysis.csv",index=False)

per_question_rows=[]
for r in FULL_ROWS:
    rm=r["retrieval_metrics"]
    per_question_rows.append({"qid":r["qid"],"question":r["question"],"gold":" | ".join(r["gold_answers"]),"prediction":r["final_answer"],
      "EM":r["exact_match"],"token_F1":r["token_f1"],"candidate_gold_R1":rm.get("recall@1"),"candidate_gold_R3":rm.get("recall@3"),
      "candidate_gold_R5":rm.get("recall@5"),"candidate_gold_R10":rm.get("recall@10"),"context_gold_found":rm.get("gold_in_context"),
      "planner_parse":r["planner"]["planner_parse_success"],"stage1_schema_parse_rate":stage_health([r],"stage1")["rate"],
      "stage1_usable_rate":stage1_usable_health([r])["rate"],"stage2_parse_rate":stage_health([r],"stage2")["rate"],"verifier_parse":stage_health([r],"verifier")["rate"],
      "verifier_status":((r.get("verification") or {}).get("parsed_output") or {}).get("status","verifier_error"),
      "feedback_used":r.get("retrieval_rounds")==2,"aggregator_parse":stage_health([r],"aggregator")["rate"],
      "failure_type":classify_failure(r),"total_seconds":r["timing"]["total_seconds"],"qwen_calls":r["qwen_call_count"],"repair_calls":r["format_repair_call_count"]})
pd.DataFrame(per_question_rows).to_csv(DIRS["metrics"]/"per_question_FULL.csv",index=False)

summary_rows=[{"method":"FULL","n":len(FULL_ROWS),"exact_correct":sum(r["exact_match"] for r in FULL_ROWS),
 "EM":mean_valid([r["exact_match"] for r in FULL_ROWS]),"mean_token_F1":mean_valid([r["token_f1"] for r in FULL_ROWS]),
 "abstention_count":sum(r["abstain"] for r in FULL_ROWS),"abstention_rate":mean_valid([r["abstain"] for r in FULL_ROWS])}]
pd.DataFrame(summary_rows).to_csv(DIRS["metrics"]/"summary.csv",index=False)
retrieval_summary_rows=[{"method":"FULL","R@1":mean_valid([r["retrieval_metrics"].get("recall@1") for r in FULL_ROWS]),
 "R@3":mean_valid([r["retrieval_metrics"].get("recall@3") for r in FULL_ROWS]),"R@5":mean_valid([r["retrieval_metrics"].get("recall@5") for r in FULL_ROWS]),
 "R@10":mean_valid([r["retrieval_metrics"].get("recall@10") for r in FULL_ROWS]),"MRR":mean_valid([r["retrieval_metrics"].get("mrr") for r in FULL_ROWS]),
 "candidate_gold_coverage":mean_valid([r["retrieval_metrics"].get("gold_candidate_pool_coverage") for r in FULL_ROWS]),
 "reasoning_context_gold_coverage":mean_valid([r["retrieval_metrics"].get("gold_context_coverage") for r in FULL_ROWS])}]
pd.DataFrame(retrieval_summary_rows).to_csv(DIRS["metrics"]/"retrieval_summary.csv",index=False)

health={"question_count":len(FULL_ROWS),"planner":stage_health(FULL_ROWS,"planner"),"stage1_schema":stage_health(FULL_ROWS,"stage1"),
        "stage1_usable":stage1_usable_health(FULL_ROWS),"stage2":stage_health(FULL_ROWS,"stage2"),"verifier":stage_health(FULL_ROWS,"verifier"),"aggregator":stage_health(FULL_ROWS,"aggregator"),
        "qwen_calls_per_question":[r["qwen_call_count"] for r in FULL_ROWS],"repair_calls_per_question":[r["format_repair_call_count"] for r in FULL_ROWS],
        "question_seconds":[r["timing"]["total_seconds"] for r in FULL_ROWS],"startup_seconds":STARTUP_SECONDS,"startup_timing":STARTUP_TIMINGS,"cache_status":CACHE_RESOURCE_STATUS,"inference_seconds":INFERENCE_SECONDS,
        "oom_events":sum(bool(x.get("oom") or x.get("oom_retried")) for x in read_jsonl_checked(GENERATION_DIAGNOSTICS_PATH)),
        "checkpoint_count":len(read_jsonl_checked(RESULT_PATHS["FULL"])),"protected_prompt_truncations":sum(bool(x.get("protected_prompt_truncated")) for x in read_jsonl_checked(GENERATION_DIAGNOSTICS_PATH))}
health["average_qwen_calls"]=mean_valid(health["qwen_calls_per_question"]); health["average_repair_calls"]=mean_valid(health["repair_calls_per_question"]); health["average_question_seconds"]=mean_valid(health["question_seconds"])

READY_FOR_16=(FAST_VALIDATION and len(FULL_ROWS)==2 and health["checkpoint_count"]==2 and health["oom_events"]==0 and
              health["protected_prompt_truncations"]==0 and health["average_repair_calls"]<=1 and
              health["average_qwen_calls"]<=13 and health["average_question_seconds"]<=120 and FEEDBACK_UNIT_TESTS_PASS and
              health["planner"]["rate"]==1.0 and health["stage1_usable"]["rate"]==1.0 and
              all(health[k]["rate"]==1.0 for k in ["stage2","verifier","aggregator"]) and
              not any(r.get("final_decision",{}).get("infrastructure_failure") or r.get("forced_parser_abstain") for r in FULL_ROWS))
health["READY_FOR_16"]=READY_FOR_16
(DIRS["metrics"]/"pipeline_health.json").write_text(json.dumps(health,indent=2,allow_nan=True),encoding="utf-8")

runtime_rows=[{"qid":r["qid"],"final_answer":r["final_answer"],"total_seconds":r["timing"]["total_seconds"],"qwen_calls":r["qwen_call_count"],
               "repair_calls":r["format_repair_call_count"],"planner_parse":r["planner"]["planner_parse_success"],
               "stage1_schema_parse_rate":stage_health([r],"stage1")["rate"],"stage1_usable_rate":stage1_usable_health([r])["rate"],"stage2_parse_rate":stage_health([r],"stage2")["rate"],
               "verifier_parse":stage_health([r],"verifier")["rate"],"aggregator_parse":stage_health([r],"aggregator")["rate"]} for r in FULL_ROWS]
pd.DataFrame(runtime_rows).to_csv(DIRS["metrics"]/"fast_validation_runtime.csv",index=False)
summary={"mode":"FAST_VALIDATION_NOT_FOR_ACCURACY" if FAST_VALIDATION else "SMOKE_16","n":len(FULL_ROWS),"startup_seconds":STARTUP_SECONDS,
         "inference_seconds":INFERENCE_SECONDS,"total_runtime_seconds":STARTUP_SECONDS+INFERENCE_SECONDS,
         "device_placement":{"qwen":str(getattr(model,"hf_device_map",{})),"bge":BGE_DEVICE,"clip_actual":CLIP.actual_device},"startup_timing":STARTUP_TIMINGS,"cache_status":CACHE_RESOURCE_STATUS,
         "health":health,"questions":runtime_rows}
prior=json.loads(OPTIMIZATION_SUMMARY_PATH.read_text()); prior.update({"runtime_result":summary,"ready_for_16":READY_FOR_16}); OPTIMIZATION_SUMMARY_PATH.write_text(json.dumps(prior,indent=2,allow_nan=True),encoding="utf-8")

print("\n==================================================\nREADY-FOR-16 VALIDATION\n==================================================")
print("Questions:",f"{len(FULL_ROWS)}/2")
print("Planner: parsed",health["planner"]["parsed"],"/",health["planner"]["total"])
print("Stage-I schema:")
for mod in ["text","table","image"]:
    item=modality_health(FULL_ROWS,"stage1",mod,lambda x:x.get("schema_parse_success",x.get("parse_success")))
    print(mod,item["passed"],"/",item["total"])
print("overall",health["stage1_schema"]["parsed"],"/",health["stage1_schema"]["total"])
print("Stage-I usable:")
for mod in ["text","table","image"]:
    item=modality_health(FULL_ROWS,"stage1",mod,lambda x:x.get("usable_output"))
    print(mod,item["passed"],"/",item["total"])
print("overall",health["stage1_usable"]["usable"],"/",health["stage1_usable"]["total"])
print("Stage-II:")
for mod in ["text","table","image"]:
    item=modality_health(FULL_ROWS,"stage2",mod,lambda x:x.get("parse_success"))
    print(f"{mod}-anchor",item["passed"],"/",item["total"])
print("overall",health["stage2"]["parsed"],"/",health["stage2"]["total"])
statuses=[((r.get("verification") or {}).get("parsed_output") or {}).get("status","verifier_error") for r in FULL_ROWS]
print("Verifier: parsed",health["verifier"]["parsed"],"/",health["verifier"]["total"],"statuses =",statuses)
print("Feedback unit tests:","PASS" if FEEDBACK_UNIT_TESTS_PASS else "FAIL")
print("Aggregator: parsed",health["aggregator"]["parsed"],"/",health["aggregator"]["total"])
print("OOM:",health["oom_events"])
print("Protected truncations:",health["protected_prompt_truncations"])
print("Runtime:")
for row in FULL_ROWS:print(row["qid"],"=",round(row["timing"]["total_seconds"],1))
print("average =",round(health["average_question_seconds"],1))
print("Qwen calls:",health["qwen_calls_per_question"])
print("Repair calls:",health["repair_calls_per_question"])
print("Startup timing:",json.dumps(STARTUP_TIMINGS,indent=2))
print("Cache:")
for label,key in [("image_manifest","image_manifest"),("text_index","text"),("table_index","table"),("image_index","image_titles")]:print(label,"=",CACHE_RESOURCE_STATUS[key])
print("Predictions:")
for row in FULL_ROWS:print(row["qid"],"→",row["final_answer"])
print("\nINFORMATION PRESERVATION TRACE")
for row in FULL_ROWS:
    trace=row.get("information_trace",{})
    print("qid",row["qid"])
    print("Stage-I raw outputs:",repr(trace.get("stage1_raw_outputs",{})))
    print("Stage-II anchor inputs:",repr(trace.get("stage2_input_anchor_text",{})))
    print("Stage-II outputs:",repr(trace.get("stage2_outputs",{})))
    print("Verifier input candidate syntheses:",repr(trace.get("verifier_input_candidate_syntheses",{})))
    print("Verifier status:",trace.get("verifier_status"))
    print("Aggregator input syntheses:",repr(trace.get("aggregator_input_syntheses",{})))
    print("Final answer:",trace.get("final_answer"))
print("\nREADY_FOR_16 =",READY_FOR_16)
if READY_FOR_16:
    print("NEXT STEP: Set FAST_VALIDATION = False and manually run the same FULL pipeline on the exact frozen 16 qids.")
else:
    failures=[]
    if health["planner"]["rate"]!=1.0:failures.append("planner_parse")
    if health["stage1_usable"]["rate"]!=1.0:failures.append("stage1_usable")
    for key in ["stage2","verifier","aggregator"]:
        if health[key]["rate"]!=1.0:failures.append(f"{key}_parse")
    if health["oom_events"]:failures.append("oom")
    if health["protected_prompt_truncations"]:failures.append("protected_prompt_truncation")
    if health["average_question_seconds"]>120:failures.append("runtime")
    print("STOP. Failing components:",failures)
print("The notebook intentionally stops after two questions; it never starts the 16-question run automatically.")


READY-FOR-16 VALIDATION
Questions: 16/2
Planner: parsed 16 / 16
Stage-I schema:
text 0 / 16
table 0 / 16
image 0 / 16
overall 0 / 48
Stage-I usable:
text 16 / 16
table 16 / 16
image 16 / 16
overall 48 / 48
Stage-II:
text-anchor 16 / 16
table-anchor 16 / 16
image-anchor 16 / 16
overall 48 / 48
Verifier: parsed 16 / 16 statuses = ['insufficient', 'insufficient', 'insufficient', 'insufficient', 'insufficient', 'insufficient', 'insufficient', 'insufficient', 'insufficient', 'insufficient', 'insufficient', 'insufficient', 'insufficient', 'insufficient', 'insufficient', 'insufficient']
Feedback unit tests: PASS
Aggregator: parsed 16 / 16
OOM: 0
Protected truncations: 0
Runtime:
f7dd1ac143507bd2ab9a480f18b86706 = 60.9
254afe636a34c3a5d94efa22d95465ad = 55.6
ebb69554283456251305b4b8ab579f76 = 39.9
4b9a4868455f98a69a8494ef1dfe7205 = 54.4
186146ea1f5a9f7ada7f7fd31c5c75a4 = 37.6
c9fcae0ac6e3aa3f08375db9e4fe9037 = 47.5
5dfd4afc175cdebf43295316a540d70c = 53.9
6d372c501004c3e448b1991f39160e99 = 52.